# RAJA Performance Suite Clustering Analysis

In this notebook, we compose performance data from experiments using the RAJA Performance Suite on various hardware:

- AMD MI-250X GPUs on LLNL's Tioga system
- NVIDIA V100 GPUs on LLNL's Lassen system
- Intel Sapphire Rapids CPUs, DDR nodes on LLNL's Poodle
- Intel Sapphire Rapids CPUS, HBM nodes on LLNL's Poodle

We reproduce some of the analysis and visualizations from the paper:

Olga Pearce, Jason Burmark, Rich Hornung, Befikir Bogale, Ian Lumsden, Michael McKinsey, Dewi Yokelson, David Boehme, 
Stephanie Brink, Michela Taufer, and Tom Scogland. “RAJA Performance Suite: Performance Portability Analysis with 
Caliper and Thicket”. SC-W 2024: Workshops of ACM/IEEE International Conference for High Performance Computing, 
Networking, Storage, and Analysis. Performance, Portability & Productivity in HPC. 2024.

## 1. Import Necessary Packages

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from glob import glob
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.cluster as sk
import plotly.graph_objects as go
from scipy.cluster.hierarchy import dendrogram
from scipy import stats

import thicket as tt

## 2. Define Kernels to Analyze

Define the sets of kernels to select for each tuning, separated by mi250x/v100 and Sapphire Rapids.

In [2]:
gpu_kernels = {
    "block_256": [
        "Algorithm_ATOMIC",
        "Algorithm_MEMCPY",
        "Algorithm_MEMSET",
        "Apps_DEL_DOT_VEC_2D",
        "Apps_EDGE3D",
        "Apps_ENERGY",
        "Apps_FIR",
        "Apps_LTIMES",
        "Apps_LTIMES_NOVIEW",
        "Apps_MATVEC_3D_STENCIL",
        "Apps_NODAL_ACCUMULATION_3D",
        "Apps_PRESSURE",
        "Apps_VOL3D",
        "Apps_ZONAL_ACCUMULATION_3D",
        "Basic_ARRAY_OF_PTRS",
        "Basic_COPY8",
        "Basic_DAXPY",
        "Basic_DAXPY_ATOMIC",
        "Basic_IF_QUAD",
        "Basic_INDEXLIST",
        "Basic_INDEXLIST_3LOOP",
        "Basic_INIT3",
        "Basic_INIT_VIEW1D",
        "Basic_INIT_VIEW1D_OFFSET",
        "Basic_MAT_MAT_SHARED",
        "Basic_MULADDSUB",
        "Basic_NESTED_INIT",
        "Basic_PI_ATOMIC",
        "Comm_HALO_EXCHANGE",
        "Comm_HALO_PACKING",
        "Comm_HALO_SENDRECV",
        "Lcals_DIFF_PREDICT",
        "Lcals_EOS",
        "Lcals_FIRST_DIFF",
        "Lcals_FIRST_SUM",
        "Lcals_GEN_LIN_RECUR",
        "Lcals_HYDRO_1D",
        "Lcals_HYDRO_2D",
        "Lcals_INT_PREDICT",
        "Lcals_PLANCKIAN",
        "Lcals_TRIDIAG_ELIM",
        "Polybench_2MM",
        "Polybench_3MM",
        "Polybench_ADI",
        "Polybench_ATAX",
        "Polybench_FDTD_2D",
        "Polybench_FLOYD_WARSHALL",
        "Polybench_GEMM",
        "Polybench_GEMVER",
        "Polybench_GESUMMV",
        "Polybench_HEAT_3D",
        "Polybench_JACOBI_1D",
        "Polybench_JACOBI_2D",
        "Polybench_MVT",
        "Stream_ADD",
        "Stream_COPY",
        "Stream_MUL",
        "Stream_TRIAD",
    ], # For block_256
    # "default": [
    #     "Algorithm_SORT",
    #     "Algorithm_SORTPAIRS",
    # ], # For default
#     "blkatm_occgs_256": [
#         "Algorithm_REDUCE_SUM",
#         "Basic_PI_REDUCE",
#         "Basic_REDUCE3_INT",
#         "Basic_REDUCE_STRUCT",
#         "Basic_TRAP_INT",
# #    ], "blkatm_direct_256": [
#         "Stream_DOT",
#     ], # For blkatm_occgs_256
    "block_64": [
        "Apps_CONVECTION3DPA",
        "Apps_DIFFUSION3DPA",
        "Apps_MASS3DEA",
    ], # For block_64
    "block_25": [
        "Apps_MASS3DPA",
    ], # For block_25
    # "funcptr_256": [
    #     "Comm_HALO_EXCHANGE_FUSED",
    #     "Comm_HALO_PACKING_FUSED",
    # ], # For funcptr_256
    # "rocprim": [
    #     "Algorithm_SCAN",
    # ], # For rocprim
#     "atomic_occgs_256": [
#         "Algorithm_HISTOGRAM",
# #    ], "atomic_direct_256": [
#         "Basic_MULTI_REDUCE",
#     ], # For atomic_occgs_256
    # "blkdev_occgs_256": [
    #     "Lcals_FIRST_MIN",
    # ], # For blkdev_occgs_256
    # "cub": [
    #     "Algorithm_SCAN",
    # ], # For cub
}

spr_kernels = {
    "default": [
        "Algorithm_ATOMIC",
        "Algorithm_HISTOGRAM",
        "Algorithm_MEMCPY",
        "Algorithm_MEMSET",
        "Algorithm_SCAN",
        "Algorithm_SORT",
        "Algorithm_SORTPAIRS",
        "Algorithm_REDUCE_SUM",
        "Apps_CONVECTION3DPA",
        "Apps_DEL_DOT_VEC_2D",
        "Apps_DIFFUSION3DPA",
        "Apps_EDGE3D",
        "Apps_ENERGY",
        "Apps_FIR",
        "Apps_LTIMES",
        "Apps_LTIMES_NOVIEW",
        "Apps_MASS3DEA",
        "Apps_MASS3DPA",
        "Apps_MATVEC_3D_STENCIL",
        "Apps_NODAL_ACCUMULATION_3D",
        "Apps_PRESSURE",
        "Apps_VOL3D",
        "Apps_ZONAL_ACCUMULATION_3D",
        "Basic_ARRAY_OF_PTRS",
        "Basic_COPY8",
        "Basic_DAXPY",
        "Basic_DAXPY_ATOMIC",
        "Basic_IF_QUAD",
        "Basic_INDEXLIST",
        "Basic_INDEXLIST_3LOOP",
        "Basic_INIT3",
        "Basic_INIT_VIEW1D",
        "Basic_INIT_VIEW1D_OFFSET",
        "Basic_MAT_MAT_SHARED",
        "Basic_MULADDSUB",
        "Basic_MULTI_REDUCE",
        "Basic_NESTED_INIT",
        "Basic_PI_ATOMIC",
        "Basic_PI_REDUCE",
        "Basic_REDUCE_STRUCT",
        "Basic_REDUCE3_INT",
        "Basic_TRAP_INT",
        "Comm_HALO_EXCHANGE",
        "Comm_HALO_PACKING",
        "Comm_HALO_SENDRECV",
        "Lcals_DIFF_PREDICT",
        "Lcals_EOS",
        "Lcals_FIRST_DIFF",
        "Lcals_FIRST_MIN",
        "Lcals_FIRST_SUM",
        "Lcals_GEN_LIN_RECUR",
        "Lcals_HYDRO_1D",
        "Lcals_HYDRO_2D",
        "Lcals_INT_PREDICT",
        "Lcals_PLANCKIAN",
        "Lcals_TRIDIAG_ELIM",
        "Polybench_2MM",
        "Polybench_3MM",
        "Polybench_ADI",
        "Polybench_ATAX",
        "Polybench_FDTD_2D",
        "Polybench_FLOYD_WARSHALL",
        "Polybench_GEMM",
        "Polybench_GEMVER",
        "Polybench_GESUMMV",
        "Polybench_HEAT_3D",
        "Polybench_JACOBI_1D",
        "Polybench_JACOBI_2D",
        "Polybench_MVT",
        "Stream_ADD",
        "Stream_COPY",
        "Stream_DOT",
        "Stream_MUL",
        "Stream_TRIAD",
    ], # default
    # "funcptr": [
    #     "Comm_HALO_EXCHANGE_FUSED",
    #     "Comm_HALO_PACKING_FUSED",
    # ], # For funcptr
}

# List of kernels to exclude from the data, agnostic of parameters
exclude_kernels = [
    "Algorithm_ATOMIC",
    "Algorithm_HISTOGRAM",
    "Algorithm_REDUCE_SUM",
    "Algorithm_SCAN",
    "Algorithm_SORT",
    "Algorithm_SORTPAIRS",

    "Apps_NODAL_ACCUMULATION_3D",

    'Basic_DAXPY_ATOMIC',
    "Basic_INDEXLIST",
    "Basic_INDEXLIST_3LOOP", 
    "Basic_MULTI_REDUCE",
    "Basic_PI_ATOMIC",
    "Basic_PI_REDUCE",
    "Basic_REDUCE_STRUCT",
    "Basic_REDUCE3_INT",
    "Basic_TRAP_INT",
    
    "Lcals_FIRST_MIN",

    "Comm_HALO_EXCHANGE",
    "Comm_HALO_EXCHANGE_FUSED",
    "Comm_HALO_SENDRECV",
    "Comm_HALO_EXCHANGE_FUSED",
    "Comm_HALO_PACKING",
    "Comm_HALO_PACKING_FUSED",

    "Polybench_GEMM",
    "Polybench_GEMVER",
    "Polybench_GESUMMV",

    "Stream_DOT",
]
Non_On_kernels = [
    "Algorithm_SORT",
    "Algorithm_SORTPAIRS",
    "Comm_HALO_EXCHANGE",
    "Comm_HALO_EXCHANGE_FUSED",
    "Comm_HALO_PACKING",
    "Comm_HALO_PACKING_FUSED",
    "Comm_HALO_SENDRECV",
    "Polybench_2MM",
    "Polybench_3MM" ,
    "Polybench_FLOYD_WARSHALL",
    "Polybench_GEMM",
    "Apps_EDGE3D",
    "Basic_PI_REDUCE",
]
exclude_kernels = exclude_kernels + Non_On_kernels

Memory_bound = ['Algorithm_MEMCPY',
 'Algorithm_MEMSET',
 'Apps_ENERGY',
 'Apps_MATVEC_3D_STENCIL',
 'Apps_PRESSURE',
 'Apps_ZONAL_ACCUMULATION_3D',
 'Basic_ARRAY_OF_PTRS',
 'Basic_COPY8',
 'Basic_DAXPY',
 'Basic_IF_QUAD',
 'Basic_INIT3',
 'Basic_MULADDSUB',
 'Lcals_DIFF_PREDICT',
 'Lcals_EOS',
 'Lcals_FIRST_DIFF',
 'Lcals_FIRST_SUM',
 'Lcals_GEN_LIN_RECUR',
 'Lcals_HYDRO_1D',
 'Lcals_HYDRO_2D',
 'Lcals_INT_PREDICT',
 'Lcals_TRIDIAG_ELIM',
 'Polybench_ADI',
 'Polybench_FDTD_2D',
 'Polybench_JACOBI_1D',
 'Polybench_JACOBI_2D',
 'Stream_ADD',
 'Stream_COPY',
 'Stream_MUL',
 'Stream_TRIAD']

retiring_bound = ['Apps_CONVECTION3DPA',
 'Apps_DEL_DOT_VEC_2D',
 'Apps_DIFFUSION3DPA',
 'Apps_FIR',
 'Apps_LTIMES',
 'Apps_LTIMES_NOVIEW',
 'Apps_MASS3DEA',
 'Apps_MASS3DPA',
 'Apps_VOL3D',
 'Basic_INIT_VIEW1D',
 'Basic_INIT_VIEW1D_OFFSET',
 'Basic_MAT_MAT_SHARED',
 'Basic_NESTED_INIT',
 'Lcals_PLANCKIAN',
 'Polybench_ATAX',
 'Polybench_HEAT_3D',
 'Polybench_MVT']

In [3]:
def kernel_query(kernel_list):
    return tt.query.Query().match(
        ".",
        lambda row: row["name"].apply(
            lambda n: n in kernel_list
        ).all()
    ).rel("*")

def not_kernel_query(kernel_list):
    return tt.query.Query().match(
        ".",
        lambda row: row["name"].apply(
            lambda n: n not in kernel_list
        ).all()
    ).rel("*")

## 3. Read Performance Data into Thicket

In [ ]:
# Directories
main_dir = "../data/topdown_metrics/"
epyc_mi250x_dir = main_dir + "epyc-mi250x/"
p9_v100_dir = main_dir + "p9-v100/"
spr_ddr_dir = main_dir + "spr-ddr/"
spr_hbm_dir = main_dir + "spr-hbm/"
epyc_mi300a_dir = main_dir + "epyc-mi300a/rzadams_cray-mpich-8.1.31-amdclang-6.3.0-gfx942_caliper/SIZE_008388608/"
cas_a100_dir = main_dir + "cas-a100/"
gr_gh200_dir = main_dir + "gr-gh200/32M/"

tk_mi250x = tt.Thicket.from_caliperreader(glob(epyc_mi250x_dir + "**/*.cali", recursive=True), fill_perfdata=False)
tk_v100 = tt.Thicket.from_caliperreader(glob(p9_v100_dir + "**/*.cali", recursive=True), fill_perfdata=False)
tk_spr_ddr = tt.Thicket.from_caliperreader(glob(spr_ddr_dir + "**/*.cali", recursive=True), fill_perfdata=False)
tk_spr_hbm = tt.Thicket.from_caliperreader(glob(spr_hbm_dir + "**/*.cali", recursive=True), fill_perfdata=False)
tk_mi300a = tt.Thicket.from_caliperreader(glob(epyc_mi300a_dir + "*.cali", recursive=True), fill_perfdata=False)
tk_a100 = tt.Thicket.from_caliperreader(glob(cas_a100_dir + "**/*.cali", recursive=True), fill_perfdata=False)
tk_gh200 = tt.Thicket.from_caliperreader(glob(gr_gh200_dir + "**/*.cali", recursive=True), fill_perfdata=False)

print("epyc-mi250x: ", len(tk_mi250x.profile), " files")
print("p9-v100: ", len(tk_v100.profile), " files")
print("cas-a100: ", len(tk_a100.profile), " files")
print("spr-ddr: ", len(tk_spr_ddr.profile), " files")
print("spr-hbm: ", len(tk_spr_hbm.profile), " files")
print("epyc-mi300a: ", len(tk_mi300a.profile), " files")
print("gr-gh200: ", len(tk_gh200.profile), " files")

## 4. Compose Performance Data in Thicket

### 4.1 Groupby parameters to apply kernel queries for each tuning

In [5]:
gb_params = ["variant", "tuning"]
gb_mi250x = tk_mi250x.groupby(gb_params)
gb_mi300a = tk_mi300a.groupby(gb_params)
gb_v100 = tk_v100.groupby(gb_params)
gb_a100 = tk_a100.groupby(gb_params)
gb_gh200 = tk_gh200.groupby(gb_params)
gb_spr_ddr = tk_spr_ddr.groupby(gb_params + ["spot:topdown.all"])
gb_spr_hbm = tk_spr_hbm.groupby(gb_params + ["spot:topdown.all"])

In [ ]:
new_gb_mi250x = {}
new_gb_v100 = {}
gb_spr_ddr_topdown = {}
gb_spr_ddr_notopdown = {}
gb_spr_hbm_topdown = {}
gb_spr_hbm_notopdown = {}
new_gb_mi300a = {}
new_gb_a100 = {}
new_gb_gh200 = {}

print("v100")
for k in gb_v100.keys():
    if k[1] in gpu_kernels.keys():
        print("\t", k)
        new_gb_v100[k] = gb_v100[k].query(kernel_query(gpu_kernels[k[1]]), multi_index_mode="all").query(not_kernel_query(exclude_kernels), multi_index_mode="all")

print("spr-ddr")
for key in gb_spr_ddr.keys():
    if key[1] in spr_kernels.keys():
        print("\t", key)
        if key[2] == "true":
            gb_spr_ddr_topdown[key] = gb_spr_ddr[key].query(kernel_query(spr_kernels[key[1]]), multi_index_mode="all").query(not_kernel_query(exclude_kernels), multi_index_mode="all")
        else:
            gb_spr_ddr_notopdown[key] = gb_spr_ddr[key].query(kernel_query(spr_kernels[key[1]]), multi_index_mode="all").query(not_kernel_query(exclude_kernels), multi_index_mode="all")

print("spr-hbm")
for key in gb_spr_hbm.keys():
    if key[1] in spr_kernels.keys():
        print("\t", key)
        if key[2] == "true":
            gb_spr_hbm_topdown[key] = gb_spr_hbm[key].query(kernel_query(spr_kernels[key[1]]), multi_index_mode="all").query(not_kernel_query(exclude_kernels), multi_index_mode="all")
        else:
            gb_spr_hbm_notopdown[key] = gb_spr_hbm[key].query(kernel_query(spr_kernels[key[1]]), multi_index_mode="all").query(not_kernel_query(exclude_kernels), multi_index_mode="all")

print("mi250x")
for k in gb_mi250x.keys():
    if k[1] in gpu_kernels.keys(): 
        print("\t", k)
        new_gb_mi250x[k] = gb_mi250x[k].query(kernel_query(gpu_kernels[k[1]]), multi_index_mode="all").query(not_kernel_query(exclude_kernels), multi_index_mode="all")

print("mi300a")
for k in gb_mi300a.keys():
    if k[1] in gpu_kernels.keys() and k[0] == "RAJA_HIP": 
        print("\t", k)
        new_gb_mi300a[k] = gb_mi300a[k].query(kernel_query(gpu_kernels[k[1]]), multi_index_mode="all").query(not_kernel_query(exclude_kernels), multi_index_mode="all")

print("a100")
for k in gb_a100.keys():
    if k[1] in gpu_kernels.keys():
        print("\t", k)
        new_gb_a100[k] = gb_a100[k].query(kernel_query(gpu_kernels[k[1]]), multi_index_mode="all").query(not_kernel_query(exclude_kernels), multi_index_mode="all")
        
print("gh200")
for k in gb_gh200.keys():
    if k[1] in gpu_kernels.keys():
        print("\t", k)
        new_gb_gh200[k] = gb_gh200[k].query(kernel_query(gpu_kernels[k[1]]), multi_index_mode="all").query(not_kernel_query(exclude_kernels), multi_index_mode="all")

### 4.2 Compose Thickets after applying queries

We have separate Thickets for mi250x, v100, and the combination for Sapphire Rapids on DDR/HBM and topdown/no-topdown. We calculate statistics for each of these Thickets separately.

In [7]:
tk_mi250x = tt.Thicket.concat_thickets(list(new_gb_mi250x.values()), fill_perfdata=False)
tk_mi300a = tt.Thicket.concat_thickets(list(new_gb_mi300a.values()), fill_perfdata=False)
tk_v100 = tt.Thicket.concat_thickets(list(new_gb_v100.values()), fill_perfdata=False)
tk_a100 = tt.Thicket.concat_thickets(list(new_gb_a100.values()), fill_perfdata=False)
tk_gh200 = tt.Thicket.concat_thickets(list(new_gb_gh200.values()), fill_perfdata=False)
tk_spr_ddr_topdown = tt.Thicket.concat_thickets(list(gb_spr_ddr_topdown.values()), fill_perfdata=False)
tk_spr_ddr_notopdown = tt.Thicket.concat_thickets(list(gb_spr_ddr_notopdown.values()), fill_perfdata=False)
tk_spr_hbm_topdown = tt.Thicket.concat_thickets(list(gb_spr_hbm_topdown.values()), fill_perfdata=False)
tk_spr_hbm_notopdown = tt.Thicket.concat_thickets(list(gb_spr_hbm_notopdown.values()), fill_perfdata=False)

types = ["mi250x", "mi300a", "v100", "a100", "gh200", "ddr_topdown", "ddr", "hbm_topdown", "hbm"]
tks = [
    tk_mi250x,
    tk_mi300a,
    tk_v100,
    tk_a100,
    tk_gh200,
    tk_spr_ddr_topdown,
    tk_spr_ddr_notopdown,
    tk_spr_hbm_topdown,
    tk_spr_hbm_notopdown,
]
tkmap = dict(zip(types, tks))

## 5. Calculate Derived Metrics

### 5.1 Define metrics names

In [8]:
# Primary time metrics
ddr_no_topdown_time_metric = ("ddr", "Avg time/rank")
hbm_no_topdown_time_metric = ("hbm", "Avg time/rank")
mi250x_time_metric = ("mi250x", "Avg time/rank")
v100_time_metric = ("v100", "Avg time/rank")
mi300a_time_metric = ("mi300a", "Avg time/rank")
a100_time_metric = ("a100", "Avg time/rank")
gh200_time_metric = ("gh200", "Avg time/rank")

# RAJAPerf metrics used for Memory Bandwidth for DDR and HBM
ddr_reps = ("ddr", "Reps")
ddr_bytes_read_per_rep = ("ddr", "BytesRead/Rep")
ddr_bytes_write_per_rep = ("ddr", "BytesWritten/Rep")
ddr_bytes_atomic_write_per_rep = ("ddr", "BytesAtomicModifyWritten/Rep")
ddr_flops_per_rep = ("ddr", "Flops/Rep")
ddr_tsteps = ("ddr", "tsteps")

hbm_reps = ("hbm", "Reps")
hbm_bytes_read_per_rep = ("hbm", "BytesRead/Rep")
hbm_bytes_write_per_rep = ("hbm", "BytesWritten/Rep")
hbm_bytes_atomic_write_per_rep = ("hbm", "BytesAtomicModifyWritten/Rep")
hbm_flops_per_rep = ("hbm", "Flops/Rep")
hbm_tsteps = ("hbm", "tsteps")

mi250x_reps = ("mi250x", "Reps")
mi250x_bytes_read_per_rep = ("mi250x", "BytesRead/Rep")
mi250x_bytes_write_per_rep = ("mi250x", "BytesWritten/Rep")
mi250x_bytes_atomic_write_per_rep = ("mi250x", "BytesAtomicModifyWritten/Rep")
mi250x_flops_per_rep = ("mi250x", "Flops/Rep")
mi250x_tsteps = ("mi250x", "tsteps")

v100_reps = ("v100", "Reps")
v100_bytes_read_per_rep = ("v100", "BytesRead/Rep")
v100_bytes_write_per_rep = ("v100", "BytesWritten/Rep")
v100_bytes_atomic_write_per_rep = ("v100", "BytesAtomicModifyWritten/Rep")
v100_flops_per_rep = ("v100", "Flops/Rep")
v100_tsteps = ("v100", "tsteps")

mi300a_reps = ("mi300a", "Reps")
mi300a_bytes_read_per_rep = ("mi300a", "BytesRead/Rep")
mi300a_bytes_write_per_rep = ("mi300a", "BytesWritten/Rep")
mi300a_bytes_atomic_write_per_rep = ("mi300a", "BytesAtomicModifyWritten/Rep")
mi300a_flops_per_rep = ("mi300a", "Flops/Rep")
mi300a_tsteps = ("mi300a", "tsteps")

a100_reps = ("a100", "Reps")
a100_bytes_read_per_rep = ("a100", "BytesRead/Rep")
a100_bytes_write_per_rep = ("a100", "BytesWritten/Rep")
a100_bytes_atomic_write_per_rep = ("a100", "BytesAtomicModifyWritten/Rep")
a100_flops_per_rep = ("a100", "Flops/Rep")
a100_tsteps = ("a100", "tsteps")

gh200_reps = ("gh200", "Reps")
gh200_bytes_read_per_rep = ("gh200", "BytesRead/Rep")
gh200_bytes_write_per_rep = ("gh200", "BytesWritten/Rep")
gh200_bytes_atomic_write_per_rep = ("gh200", "BytesAtomicModifyWritten/Rep")
gh200_flops_per_rep = ("gh200", "Flops/Rep")
gh200_tsteps = ("gh200", "tsteps")

# Topdown metrics for DDR and HBM
ddr_be_bound = ("ddr_topdown", "Backend bound")
ddr_mem_bound = ("ddr_topdown", "Memory bound")
ddr_core_bound = ("ddr_topdown", "Core bound")
ddr_fe_bound = ("ddr_topdown", "Frontend bound")
ddr_fe_lat = ("ddr_topdown", "Frontend latency")
ddr_fe_bw = ("ddr_topdown", "Frontend bandwidth")
ddr_bad_spec = ("ddr_topdown", "Bad speculation")
ddr_br_mispred = ("ddr_topdown", "Branch mispredict")
ddr_machine_clears = ("ddr_topdown", "Machine clears")
ddr_retiring = ("ddr_topdown", "Retiring")
ddr_heavy_ops = ("ddr_topdown", "Heavy operations")
ddr_light_ops = ("ddr_topdown", "Light operations")

hbm_be_bound = ("hbm_topdown", "Backend bound")
hbm_mem_bound = ("hbm_topdown", "Memory bound")
hbm_core_bound = ("hbm_topdown", "Core bound")
hbm_fe_bound = ("hbm_topdown", "Frontend bound")
hbm_fe_lat = ("hbm_topdown", "Frontend latency")
hbm_fe_bw = ("hbm_topdown", "Frontend bandwidth")
hbm_bad_spec = ("hbm_topdown", "Bad speculation")
hbm_br_mispred = ("hbm_topdown", "Branch mispredict")
hbm_machine_clears = ("hbm_topdown", "Machine clears")
hbm_retiring = ("hbm_topdown", "Retiring")
hbm_heavy_ops = ("hbm_topdown", "Heavy operations")
hbm_light_ops = ("hbm_topdown", "Light operations")

ddr_problem_size = ("ddr", "ProblemSize")
hbm_problem_size = ("hbm", "ProblemSize")
mi250x_problem_size = ("mi250x", "ProblemSize")
v100_problem_size = ("v100", "ProblemSize")
mi300a_problem_size = ("mi300a", "ProblemSize")
a100_problem_size = ("a100", "ProblemSize")
gh200_problem_size = ("gh200", "ProblemSize")

# Configurable derived metrics
ddr_bytes_read_per_rep_scaled = ("ddr", "Bytes Read/Rep/Problem Size")
hbm_bytes_read_per_rep_scaled = ("hbm", "Bytes Read/Rep/Problem Size")
mi250x_bytes_read_per_rep_scaled = ("mi250x", "Bytes Read/Rep/Problem Size")
v100_bytes_read_per_rep_scaled = ("v100", "Bytes Read/Rep/Problem Size")
mi300a_bytes_read_per_rep_scaled = ("mi300a", "Bytes Read/Rep/Problem Size")
a100_bytes_read_per_rep_scaled = ("a100", "Bytes Read/Rep/Problem Size")
gh200_bytes_read_per_rep_scaled = ("gh200", "Bytes Read/Rep/Problem Size")

ddr_bytes_write_per_rep_scaled = ("ddr", "Bytes Written/Rep/Problem Size")
hbm_bytes_write_per_rep_scaled = ("hbm", "Bytes Written/Rep/Problem Size")
mi250x_bytes_write_per_rep_scaled = ("mi250x", "Bytes Written/Rep/Problem Size")
v100_bytes_write_per_rep_scaled = ("v100", "Bytes Written/Rep/Problem Size")
mi300a_bytes_write_per_rep_scaled = ("mi300a", "Bytes Written/Rep/Problem Size")
a100_bytes_write_per_rep_scaled = ("a100", "Bytes Written/Rep/Problem Size")
gh200_bytes_write_per_rep_scaled = ("gh200", "Bytes Written/Rep/Problem Size")

ddr_flops_per_rep_scaled = ("ddr", "FLOPs/Rep/Problem Size")
hbm_flops_per_rep_scaled = ("hbm", "FLOPs/Rep/Problem Size")
mi250x_flops_per_rep_scaled = ("mi250x", "FLOPs/Rep/Problem Size")
v100_flops_per_rep_scaled = ("v100", "FLOPs/Rep/Problem Size")
mi300a_flops_per_rep_scaled = ("mi300a", "FLOPs/Rep/Problem Size")
a100_flops_per_rep_scaled = ("a100", "FLOPs/Rep/Problem Size")
gh200_flops_per_rep_scaled = ("gh200", "FLOPs/Rep/Problem Size")

ddr_bytes_per_rep = ("ddr", "Bytes/Rep")
hbm_bytes_per_rep = ("hbm", "Bytes/Rep")
mi250x_bytes_per_rep = ("mi250x", "Bytes/Rep")
v100_bytes_per_rep = ("v100", "Bytes/Rep")
mi300a_bytes_per_rep = ("mi300a", "Bytes/Rep")
a100_bytes_per_rep = ("a100", "Bytes/Rep")
gh200_bytes_per_rep = ("gh200", "Bytes/Rep")

ddr_bytes_per_rep_scaled = ("ddr", "Bytes/Rep/Problem Size")
hbm_bytes_per_rep_scaled = ("hbm", "Bytes/Rep/Problem Size")
mi250x_bytes_per_rep_scaled = ("mi250x", "Bytes/Rep/Problem Size")
v100_bytes_per_rep_scaled = ("v100", "Bytes/Rep/Problem Size")
mi300a_bytes_per_rep_scaled = ("mi300a", "Bytes/Rep/Problem Size")
a100_bytes_per_rep_scaled = ("a100", "Bytes/Rep/Problem Size")
gh200_bytes_per_rep_scaled = ("gh200", "Bytes/Rep/Problem Size")

ddr_flops_per_byte = ("ddr", "FLOPs/Byte")
hbm_flops_per_byte = ("hbm", "FLOPs/Byte")
mi250x_flops_per_byte = ("mi250x", "FLOPs/Byte")
v100_flops_per_byte = ("v100", "FLOPs/Byte")
mi300a_flops_per_byte = ("mi300a", "FLOPs/Byte")
a100_flops_per_byte = ("a100", "FLOPs/Byte")
gh200_flops_per_byte = ("gh200", "FLOPs/Byte")

ddr_flop_rate = ("ddr", "FLOP rate (GFLOPS)")
hbm_flop_rate = ("hbm", "FLOP rate (GFLOPS)")
mi250x_flop_rate = ("mi250x", "FLOP rate (GFLOPS)")
v100_flop_rate = ("v100", "FLOP rate (GFLOPS)")
mi300a_flop_rate = ("mi300a", "FLOP rate (GFLOPS)")
a100_flop_rate = ("a100", "FLOP rate (GFLOPS)")
gh200_flop_rate = ("gh200", "FLOP rate (GFLOPS)")

ddr_read_bw = ("ddr", "Read Bandwidth (GB/sec)")
hbm_read_bw = ("hbm", "Read Bandwidth (GB/sec)")
mi250x_read_bw = ("mi250x", "Read Bandwidth (GB/sec)")
v100_read_bw = ("v100", "Read Bandwidth (GB/sec)")
mi300a_read_bw = ("mi300a", "Read Bandwidth (GB/sec)")
a100_read_bw = ("a100", "Read Bandwidth (GB/sec)")
gh200_read_bw = ("gh200", "Read Bandwidth (GB/sec)")

ddr_write_bw = ("ddr", "Write Bandwidth (GB/sec)")
hbm_write_bw = ("hbm", "Write Bandwidth (GB/sec)")
mi250x_write_bw = ("mi250x", "Write Bandwidth (GB/sec)")
v100_write_bw = ("v100", "Write Bandwidth (GB/sec)")
mi300a_write_bw = ("mi300a", "Write Bandwidth (GB/sec)")
a100_write_bw = ("a100", "Write Bandwidth (GB/sec)")
gh200_write_bw = ("gh200", "Write Bandwidth (GB/sec)")

ddr_mem_bw = ("ddr", "Memory Bandwidth (GB/sec)")
hbm_mem_bw = ("hbm", "Memory Bandwidth (GB/sec)")
mi250x_mem_bw = ("mi250x", "Memory Bandwidth (GB/sec)")
v100_mem_bw = ("v100", "Memory Bandwidth (GB/sec)")
mi300a_mem_bw = ("mi300a", "Memory Bandwidth (GB/sec)")
a100_mem_bw = ("a100", "Memory Bandwidth (GB/sec)")
gh200_mem_bw = ("gh200", "Memory Bandwidth (GB/sec)")

ddr_atom_bw = ("ddr", "Atomic Write Bandwidth (GB/sec)")
hbm_atom_bw = ("hbm", "Atomic Write Bandwidth (GB/sec)")
mi250x_atom_bw = ("mi250x", "Atomic Write Bandwidth (GB/sec)")
v100_atom_bw = ("v100", "Atomic Write Bandwidth (GB/sec)")
mi300a_atom_bw = ("mi300a", "Atomic Write Bandwidth (GB/sec)")
a100_atom_bw = ("a100", "Atomic Write Bandwidth (GB/sec)")
gh200_atom_bw = ("gh200", "Atomic Write Bandwidth (GB/sec)")

### 5.2 Insert manually defined timestep mapping into dataframes

In [9]:
timestep_mapping = {
    "Polybench_ADI": 4,
    "Polybench_FDTD_2D": 40,
    "Polybench_JACOBI_1D": 16,
    "Polybench_HEAT_3D": 20,
    "Polybench_JACOBI_2D": 40,
}

for k in tkmap:
    tkmap[k].dataframe["tsteps"] = tkmap[k].dataframe["name"].apply(lambda kernel_name: timestep_mapping[kernel_name] if kernel_name in timestep_mapping else 1)

### 5.3 Calculate metrics

In [10]:
# Adjust bytes read per rep by number of cores (CPU) or number of GPUs (GPU)
tkmap[ddr_bytes_read_per_rep[0]].dataframe[ddr_bytes_read_per_rep[1]] *= 112
tkmap[hbm_bytes_read_per_rep[0]].dataframe[hbm_bytes_read_per_rep[1]] *= 112
tkmap[mi250x_bytes_read_per_rep[0]].dataframe[mi250x_bytes_read_per_rep[1]] *= 8
tkmap[v100_bytes_read_per_rep[0]].dataframe[v100_bytes_read_per_rep[1]] *= 4
tkmap[mi300a_bytes_read_per_rep[0]].dataframe[mi300a_bytes_read_per_rep[1]] *= 4
tkmap[a100_bytes_read_per_rep[0]].dataframe[a100_bytes_read_per_rep[1]] *= 2
tkmap[gh200_bytes_read_per_rep[0]].dataframe[gh200_bytes_read_per_rep[1]] *= 1

# Adjust bytes written per rep by number of cores (CPU) or number of GPUs (GPU)
tkmap[ddr_bytes_write_per_rep[0]].dataframe[ddr_bytes_write_per_rep[1]] *= 112
tkmap[hbm_bytes_write_per_rep[0]].dataframe[hbm_bytes_write_per_rep[1]] *= 112
tkmap[mi250x_bytes_write_per_rep[0]].dataframe[mi250x_bytes_write_per_rep[1]] *= 8
tkmap[v100_bytes_write_per_rep[0]].dataframe[v100_bytes_write_per_rep[1]] *= 4
tkmap[mi300a_bytes_write_per_rep[0]].dataframe[mi300a_bytes_write_per_rep[1]] *= 4
tkmap[a100_bytes_write_per_rep[0]].dataframe[a100_bytes_write_per_rep[1]] *= 2
tkmap[gh200_bytes_write_per_rep[0]].dataframe[gh200_bytes_write_per_rep[1]] *= 1

# Adjust flops per rep by number of cores (CPU) or number of GPUs (GPU)
tkmap[ddr_flops_per_rep[0]].dataframe[ddr_flops_per_rep[1]] *= 112
tkmap[hbm_flops_per_rep[0]].dataframe[hbm_flops_per_rep[1]] *= 112
tkmap[mi250x_flops_per_rep[0]].dataframe[mi250x_flops_per_rep[1]] *= 8
tkmap[v100_flops_per_rep[0]].dataframe[v100_flops_per_rep[1]] *= 4
tkmap[mi300a_flops_per_rep[0]].dataframe[mi300a_flops_per_rep[1]] *= 4
tkmap[a100_flops_per_rep[0]].dataframe[a100_flops_per_rep[1]] *= 2
tkmap[gh200_flops_per_rep[0]].dataframe[gh200_flops_per_rep[1]] *= 1

# SCALED METRICS
# Calculate bytes read per rep scaled per problem per device (i.e. per core or per GPU)
tkmap[ddr_bytes_read_per_rep_scaled[0]].dataframe[ddr_bytes_read_per_rep_scaled[1]] = tkmap[ddr_bytes_read_per_rep[0]].dataframe[ddr_bytes_read_per_rep[1]] / tkmap[ddr_tsteps[0]].dataframe[ddr_tsteps[1]] / tkmap[ddr_problem_size[0]].dataframe[ddr_problem_size[1]] / 112
tkmap[hbm_bytes_read_per_rep_scaled[0]].dataframe[hbm_bytes_read_per_rep_scaled[1]] = tkmap[hbm_bytes_read_per_rep[0]].dataframe[hbm_bytes_read_per_rep[1]] / tkmap[hbm_tsteps[0]].dataframe[hbm_tsteps[1]] / tkmap[hbm_problem_size[0]].dataframe[hbm_problem_size[1]] / 112
tkmap[mi250x_bytes_read_per_rep_scaled[0]].dataframe[mi250x_bytes_read_per_rep_scaled[1]] = tkmap[mi250x_bytes_read_per_rep[0]].dataframe[mi250x_bytes_read_per_rep[1]] / tkmap[mi250x_tsteps[0]].dataframe[mi250x_tsteps[1]] / tkmap[mi250x_problem_size[0]].dataframe[mi250x_problem_size[1]] / 8
tkmap[v100_bytes_read_per_rep_scaled[0]].dataframe[v100_bytes_read_per_rep_scaled[1]] = tkmap[v100_bytes_read_per_rep[0]].dataframe[v100_bytes_read_per_rep[1]] / tkmap[v100_tsteps[0]].dataframe[v100_tsteps[1]] / tkmap[v100_problem_size[0]].dataframe[v100_problem_size[1]] / 4
tkmap[mi300a_bytes_read_per_rep_scaled[0]].dataframe[mi300a_bytes_read_per_rep_scaled[1]] = tkmap[mi300a_bytes_read_per_rep[0]].dataframe[mi300a_bytes_read_per_rep[1]] / tkmap[mi300a_tsteps[0]].dataframe[mi300a_tsteps[1]] / tkmap[mi300a_problem_size[0]].dataframe[mi300a_problem_size[1]] / 4
tkmap[a100_bytes_read_per_rep_scaled[0]].dataframe[a100_bytes_read_per_rep_scaled[1]] = tkmap[a100_bytes_read_per_rep[0]].dataframe[a100_bytes_read_per_rep[1]] / tkmap[a100_tsteps[0]].dataframe[a100_tsteps[1]] / tkmap[a100_problem_size[0]].dataframe[a100_problem_size[1]] / 2
tkmap[gh200_bytes_read_per_rep_scaled[0]].dataframe[gh200_bytes_read_per_rep_scaled[1]] = tkmap[gh200_bytes_read_per_rep[0]].dataframe[gh200_bytes_read_per_rep[1]] / tkmap[gh200_tsteps[0]].dataframe[gh200_tsteps[1]] / tkmap[gh200_problem_size[0]].dataframe[gh200_problem_size[1]] / 1
# Calculate bytes written per rep scaled per problem per device (i.e. per core or per GPU)
tkmap[ddr_bytes_write_per_rep_scaled[0]].dataframe[ddr_bytes_write_per_rep_scaled[1]] = tkmap[ddr_bytes_write_per_rep[0]].dataframe[ddr_bytes_write_per_rep[1]] / tkmap[ddr_tsteps[0]].dataframe[ddr_tsteps[1]] / tkmap[ddr_problem_size[0]].dataframe[ddr_problem_size[1]] / 112
tkmap[hbm_bytes_write_per_rep_scaled[0]].dataframe[hbm_bytes_write_per_rep_scaled[1]] = tkmap[hbm_bytes_write_per_rep[0]].dataframe[hbm_bytes_write_per_rep[1]] / tkmap[hbm_tsteps[0]].dataframe[hbm_tsteps[1]] / tkmap[hbm_problem_size[0]].dataframe[hbm_problem_size[1]] / 112
tkmap[mi250x_bytes_write_per_rep_scaled[0]].dataframe[mi250x_bytes_write_per_rep_scaled[1]] = tkmap[mi250x_bytes_write_per_rep[0]].dataframe[mi250x_bytes_write_per_rep[1]] / tkmap[mi250x_tsteps[0]].dataframe[mi250x_tsteps[1]] / tkmap[mi250x_problem_size[0]].dataframe[mi250x_problem_size[1]] / 8
tkmap[v100_bytes_write_per_rep_scaled[0]].dataframe[v100_bytes_write_per_rep_scaled[1]] = tkmap[v100_bytes_write_per_rep[0]].dataframe[v100_bytes_write_per_rep[1]] / tkmap[v100_tsteps[0]].dataframe[v100_tsteps[1]] / tkmap[v100_problem_size[0]].dataframe[v100_problem_size[1]] / 4
tkmap[mi300a_bytes_write_per_rep_scaled[0]].dataframe[mi300a_bytes_write_per_rep_scaled[1]] = tkmap[mi300a_bytes_write_per_rep[0]].dataframe[mi300a_bytes_write_per_rep[1]] / tkmap[mi300a_tsteps[0]].dataframe[mi300a_tsteps[1]] / tkmap[mi300a_problem_size[0]].dataframe[mi300a_problem_size[1]] / 4
tkmap[a100_bytes_write_per_rep_scaled[0]].dataframe[a100_bytes_write_per_rep_scaled[1]] = tkmap[a100_bytes_write_per_rep[0]].dataframe[a100_bytes_write_per_rep[1]] / tkmap[a100_tsteps[0]].dataframe[a100_tsteps[1]] / tkmap[a100_problem_size[0]].dataframe[a100_problem_size[1]] / 2
tkmap[gh200_bytes_write_per_rep_scaled[0]].dataframe[gh200_bytes_write_per_rep_scaled[1]] = tkmap[gh200_bytes_write_per_rep[0]].dataframe[gh200_bytes_write_per_rep[1]] / tkmap[gh200_tsteps[0]].dataframe[gh200_tsteps[1]] / tkmap[gh200_problem_size[0]].dataframe[gh200_problem_size[1]] / 1
# Calculate flops per rep scaled per problem per device (i.e. per core or per GPU)
tkmap[ddr_flops_per_rep_scaled[0]].dataframe[ddr_flops_per_rep_scaled[1]] = tkmap[ddr_flops_per_rep[0]].dataframe[ddr_flops_per_rep[1]] / tkmap[ddr_tsteps[0]].dataframe[ddr_tsteps[1]] / tkmap[ddr_problem_size[0]].dataframe[ddr_problem_size[1]] / 112
tkmap[hbm_flops_per_rep_scaled[0]].dataframe[hbm_flops_per_rep_scaled[1]] = tkmap[hbm_flops_per_rep[0]].dataframe[hbm_flops_per_rep[1]] / tkmap[hbm_tsteps[0]].dataframe[hbm_tsteps[1]] / tkmap[hbm_problem_size[0]].dataframe[hbm_problem_size[1]] / 112
tkmap[mi250x_flops_per_rep_scaled[0]].dataframe[mi250x_flops_per_rep_scaled[1]] = tkmap[mi250x_flops_per_rep[0]].dataframe[mi250x_flops_per_rep[1]] / tkmap[mi250x_tsteps[0]].dataframe[mi250x_tsteps[1]] / tkmap[mi250x_problem_size[0]].dataframe[mi250x_problem_size[1]] / 8
tkmap[v100_flops_per_rep_scaled[0]].dataframe[v100_flops_per_rep_scaled[1]] = tkmap[v100_flops_per_rep[0]].dataframe[v100_flops_per_rep[1]] / tkmap[v100_tsteps[0]].dataframe[v100_tsteps[1]] / tkmap[v100_problem_size[0]].dataframe[v100_problem_size[1]] / 4
tkmap[mi300a_flops_per_rep_scaled[0]].dataframe[mi300a_flops_per_rep_scaled[1]] = tkmap[mi300a_flops_per_rep[0]].dataframe[mi300a_flops_per_rep[1]] / tkmap[mi300a_tsteps[0]].dataframe[mi300a_tsteps[1]] / tkmap[mi300a_problem_size[0]].dataframe[mi300a_problem_size[1]] / 4
tkmap[a100_flops_per_rep_scaled[0]].dataframe[a100_flops_per_rep_scaled[1]] = tkmap[a100_flops_per_rep[0]].dataframe[a100_flops_per_rep[1]] / tkmap[a100_tsteps[0]].dataframe[a100_tsteps[1]] / tkmap[a100_problem_size[0]].dataframe[a100_problem_size[1]] / 2
tkmap[gh200_flops_per_rep_scaled[0]].dataframe[gh200_flops_per_rep_scaled[1]] = tkmap[gh200_flops_per_rep[0]].dataframe[gh200_flops_per_rep[1]] / tkmap[gh200_tsteps[0]].dataframe[gh200_tsteps[1]] / tkmap[gh200_problem_size[0]].dataframe[gh200_problem_size[1]] / 1
# Calculate total bytes per rep from scaled byteswritten and bytesread
tkmap[ddr_bytes_per_rep_scaled[0]].dataframe[ddr_bytes_per_rep_scaled[1]] = tkmap[ddr_bytes_read_per_rep_scaled[0]].dataframe[ddr_bytes_read_per_rep_scaled[1]] + tkmap[ddr_bytes_write_per_rep_scaled[0]].dataframe[ddr_bytes_write_per_rep_scaled[1]]
tkmap[hbm_bytes_per_rep_scaled[0]].dataframe[hbm_bytes_per_rep_scaled[1]] = tkmap[hbm_bytes_read_per_rep_scaled[0]].dataframe[hbm_bytes_read_per_rep_scaled[1]] + tkmap[hbm_bytes_write_per_rep_scaled[0]].dataframe[hbm_bytes_write_per_rep_scaled[1]]
tkmap[mi250x_bytes_per_rep_scaled[0]].dataframe[mi250x_bytes_per_rep_scaled[1]] = tkmap[mi250x_bytes_read_per_rep_scaled[0]].dataframe[mi250x_bytes_read_per_rep_scaled[1]] + tkmap[mi250x_bytes_write_per_rep_scaled[0]].dataframe[mi250x_bytes_write_per_rep_scaled[1]]
tkmap[v100_bytes_per_rep_scaled[0]].dataframe[v100_bytes_per_rep_scaled[1]] = tkmap[v100_bytes_read_per_rep_scaled[0]].dataframe[v100_bytes_read_per_rep_scaled[1]] + tkmap[v100_bytes_write_per_rep_scaled[0]].dataframe[v100_bytes_write_per_rep_scaled[1]]
tkmap[mi300a_bytes_per_rep_scaled[0]].dataframe[mi300a_bytes_per_rep_scaled[1]] = tkmap[mi300a_bytes_read_per_rep_scaled[0]].dataframe[mi300a_bytes_read_per_rep_scaled[1]] + tkmap[mi300a_bytes_write_per_rep_scaled[0]].dataframe[mi300a_bytes_write_per_rep_scaled[1]]
tkmap[a100_bytes_per_rep_scaled[0]].dataframe[a100_bytes_per_rep_scaled[1]] = tkmap[a100_bytes_read_per_rep_scaled[0]].dataframe[a100_bytes_read_per_rep_scaled[1]] + tkmap[a100_bytes_write_per_rep_scaled[0]].dataframe[a100_bytes_write_per_rep_scaled[1]]
tkmap[gh200_bytes_per_rep_scaled[0]].dataframe[gh200_bytes_per_rep_scaled[1]] = tkmap[gh200_bytes_read_per_rep_scaled[0]].dataframe[gh200_bytes_read_per_rep_scaled[1]] + tkmap[gh200_bytes_write_per_rep_scaled[0]].dataframe[gh200_bytes_write_per_rep_scaled[1]]
# Calculate flops per byte
tkmap[ddr_flops_per_byte[0]].dataframe[ddr_flops_per_byte[1]] = tkmap[ddr_flops_per_rep_scaled[0]].dataframe[ddr_flops_per_rep_scaled[1]] / tkmap[ddr_bytes_per_rep_scaled[0]].dataframe[ddr_bytes_per_rep_scaled[1]]
tkmap[hbm_flops_per_byte[0]].dataframe[hbm_flops_per_byte[1]] = tkmap[hbm_flops_per_rep_scaled[0]].dataframe[hbm_flops_per_rep_scaled[1]] / tkmap[hbm_bytes_per_rep_scaled[0]].dataframe[hbm_bytes_per_rep_scaled[1]]
tkmap[mi250x_flops_per_byte[0]].dataframe[mi250x_flops_per_byte[1]] = tkmap[mi250x_flops_per_rep_scaled[0]].dataframe[mi250x_flops_per_rep_scaled[1]] / tkmap[mi250x_bytes_per_rep_scaled[0]].dataframe[mi250x_bytes_per_rep_scaled[1]]
tkmap[v100_flops_per_byte[0]].dataframe[v100_flops_per_byte[1]] = tkmap[v100_flops_per_rep_scaled[0]].dataframe[v100_flops_per_rep_scaled[1]] / tkmap[v100_bytes_per_rep_scaled[0]].dataframe[v100_bytes_per_rep_scaled[1]]
tkmap[mi300a_flops_per_byte[0]].dataframe[mi300a_flops_per_byte[1]] = tkmap[mi300a_flops_per_rep_scaled[0]].dataframe[mi300a_flops_per_rep_scaled[1]] / tkmap[mi300a_bytes_per_rep_scaled[0]].dataframe[mi300a_bytes_per_rep_scaled[1]]
tkmap[a100_flops_per_byte[0]].dataframe[a100_flops_per_byte[1]] = tkmap[a100_flops_per_rep_scaled[0]].dataframe[a100_flops_per_rep_scaled[1]] / tkmap[a100_bytes_per_rep_scaled[0]].dataframe[a100_bytes_per_rep_scaled[1]]
tkmap[gh200_flops_per_byte[0]].dataframe[gh200_flops_per_byte[1]] = tkmap[gh200_flops_per_rep_scaled[0]].dataframe[gh200_flops_per_rep_scaled[1]] / tkmap[gh200_bytes_per_rep_scaled[0]].dataframe[gh200_bytes_per_rep_scaled[1]]

# Calculate flop rate (in GFLOPS)
tkmap[ddr_flop_rate[0]].dataframe[ddr_flop_rate[1]] = (tkmap[ddr_flops_per_rep[0]].dataframe[ddr_flops_per_rep[1]] / (tkmap[ddr_no_topdown_time_metric[0]].dataframe[ddr_no_topdown_time_metric[1]] / tkmap[ddr_reps[0]].dataframe[ddr_reps[1]])) / (10**9)
tkmap[hbm_flop_rate[0]].dataframe[hbm_flop_rate[1]] = (tkmap[hbm_flops_per_rep[0]].dataframe[hbm_flops_per_rep[1]] / (tkmap[hbm_no_topdown_time_metric[0]].dataframe[hbm_no_topdown_time_metric[1]] / tkmap[hbm_reps[0]].dataframe[hbm_reps[1]])) / (10**9)
tkmap[mi250x_flop_rate[0]].dataframe[mi250x_flop_rate[1]] = (tkmap[mi250x_flops_per_rep[0]].dataframe[mi250x_flops_per_rep[1]] / (tkmap[mi250x_time_metric[0]].dataframe[mi250x_time_metric[1]] / tkmap[mi250x_reps[0]].dataframe[mi250x_reps[1]])) / (10**9)
tkmap[v100_flop_rate[0]].dataframe[v100_flop_rate[1]] = (tkmap[v100_flops_per_rep[0]].dataframe[v100_flops_per_rep[1]] / (tkmap[v100_time_metric[0]].dataframe[v100_time_metric[1]] / tkmap[v100_reps[0]].dataframe[v100_reps[1]])) / (10**9)
tkmap[mi300a_flop_rate[0]].dataframe[mi300a_flop_rate[1]] = (tkmap[mi300a_flops_per_rep[0]].dataframe[mi300a_flops_per_rep[1]] / (tkmap[mi300a_time_metric[0]].dataframe[mi300a_time_metric[1]] / tkmap[mi300a_reps[0]].dataframe[mi300a_reps[1]])) / (10**9)
tkmap[a100_flop_rate[0]].dataframe[a100_flop_rate[1]] = (tkmap[a100_flops_per_rep[0]].dataframe[a100_flops_per_rep[1]] / (tkmap[a100_time_metric[0]].dataframe[a100_time_metric[1]] / tkmap[a100_reps[0]].dataframe[a100_reps[1]])) / (10**9)
tkmap[gh200_flop_rate[0]].dataframe[gh200_flop_rate[1]] = (tkmap[gh200_flops_per_rep[0]].dataframe[gh200_flops_per_rep[1]] / (tkmap[gh200_time_metric[0]].dataframe[gh200_time_metric[1]] / tkmap[gh200_reps[0]].dataframe[gh200_reps[1]])) / (10**9)

# Calculate read bandwidth (in GB/sec)
tkmap[ddr_read_bw[0]].dataframe[ddr_read_bw[1]] = ((tkmap[ddr_bytes_read_per_rep[0]].dataframe[ddr_bytes_read_per_rep[1]]) / (tkmap[ddr_no_topdown_time_metric[0]].dataframe[ddr_no_topdown_time_metric[1]] / tkmap[ddr_reps[0]].dataframe[ddr_reps[1]])) / (10**9)
tkmap[hbm_read_bw[0]].dataframe[hbm_read_bw[1]] = ((tkmap[hbm_bytes_read_per_rep[0]].dataframe[hbm_bytes_read_per_rep[1]]) / (tkmap[hbm_no_topdown_time_metric[0]].dataframe[hbm_no_topdown_time_metric[1]] / tkmap[hbm_reps[0]].dataframe[hbm_reps[1]])) / (10**9)
tkmap[mi250x_read_bw[0]].dataframe[mi250x_read_bw[1]] = ((tkmap[mi250x_bytes_read_per_rep[0]].dataframe[mi250x_bytes_read_per_rep[1]]) / (tkmap[mi250x_time_metric[0]].dataframe[mi250x_time_metric[1]] / tkmap[mi250x_reps[0]].dataframe[mi250x_reps[1]])) / (10**9)
tkmap[v100_read_bw[0]].dataframe[v100_read_bw[1]] = ((tkmap[v100_bytes_read_per_rep[0]].dataframe[v100_bytes_read_per_rep[1]]) / (tkmap[v100_time_metric[0]].dataframe[v100_time_metric[1]] / tkmap[v100_reps[0]].dataframe[v100_reps[1]])) / (10**9)
tkmap[mi300a_read_bw[0]].dataframe[mi300a_read_bw[1]] = ((tkmap[mi300a_bytes_read_per_rep[0]].dataframe[mi300a_bytes_read_per_rep[1]]) / (tkmap[mi300a_time_metric[0]].dataframe[mi300a_time_metric[1]] / tkmap[mi300a_reps[0]].dataframe[mi300a_reps[1]])) / (10**9)
tkmap[a100_read_bw[0]].dataframe[a100_read_bw[1]] = ((tkmap[a100_bytes_read_per_rep[0]].dataframe[a100_bytes_read_per_rep[1]]) / (tkmap[a100_time_metric[0]].dataframe[a100_time_metric[1]] / tkmap[a100_reps[0]].dataframe[a100_reps[1]])) / (10**9)
tkmap[gh200_read_bw[0]].dataframe[gh200_read_bw[1]] = ((tkmap[gh200_bytes_read_per_rep[0]].dataframe[gh200_bytes_read_per_rep[1]]) / (tkmap[gh200_time_metric[0]].dataframe[gh200_time_metric[1]] / tkmap[gh200_reps[0]].dataframe[gh200_reps[1]])) / (10**9)

# Calculate write bandwidth (in GB/sec)
tkmap[ddr_write_bw[0]].dataframe[ddr_write_bw[1]] = ((tkmap[ddr_bytes_write_per_rep[0]].dataframe[ddr_bytes_write_per_rep[1]]) / (tkmap[ddr_no_topdown_time_metric[0]].dataframe[ddr_no_topdown_time_metric[1]] / tkmap[ddr_reps[0]].dataframe[ddr_reps[1]])) / (10**9)# Calculate flops per byte
tkmap[hbm_write_bw[0]].dataframe[hbm_write_bw[1]] = ((tkmap[hbm_bytes_write_per_rep[0]].dataframe[hbm_bytes_write_per_rep[1]]) / (tkmap[hbm_no_topdown_time_metric[0]].dataframe[hbm_no_topdown_time_metric[1]] / tkmap[hbm_reps[0]].dataframe[hbm_reps[1]])) / (10**9)
tkmap[mi250x_write_bw[0]].dataframe[mi250x_write_bw[1]] = ((tkmap[mi250x_bytes_write_per_rep[0]].dataframe[mi250x_bytes_write_per_rep[1]]) / (tkmap[mi250x_time_metric[0]].dataframe[mi250x_time_metric[1]] / tkmap[mi250x_reps[0]].dataframe[mi250x_reps[1]])) / (10**9)
tkmap[v100_write_bw[0]].dataframe[v100_write_bw[1]] = ((tkmap[v100_bytes_write_per_rep[0]].dataframe[v100_bytes_write_per_rep[1]]) / (tkmap[v100_time_metric[0]].dataframe[v100_time_metric[1]] / tkmap[v100_reps[0]].dataframe[v100_reps[1]])) / (10**9)   
tkmap[mi300a_write_bw[0]].dataframe[mi300a_write_bw[1]] = ((tkmap[mi300a_bytes_write_per_rep[0]].dataframe[mi300a_bytes_write_per_rep[1]]) / (tkmap[mi300a_time_metric[0]].dataframe[mi300a_time_metric[1]] / tkmap[mi300a_reps[0]].dataframe[mi300a_reps[1]])) / (10**9)
tkmap[a100_write_bw[0]].dataframe[a100_write_bw[1]] = ((tkmap[a100_bytes_write_per_rep[0]].dataframe[a100_bytes_write_per_rep[1]]) / (tkmap[a100_time_metric[0]].dataframe[a100_time_metric[1]] / tkmap[a100_reps[0]].dataframe[a100_reps[1]])) / (10**9)
tkmap[gh200_write_bw[0]].dataframe[gh200_write_bw[1]] = ((tkmap[gh200_bytes_write_per_rep[0]].dataframe[gh200_bytes_write_per_rep[1]]) / (tkmap[gh200_time_metric[0]].dataframe[gh200_time_metric[1]] / tkmap[gh200_reps[0]].dataframe[gh200_reps[1]])) / (10**9)

# Calculate total memory bandwidth (in GB/sec)
tkmap[ddr_mem_bw[0]].dataframe[ddr_mem_bw[1]] = ((tkmap[ddr_bytes_read_per_rep[0]].dataframe[ddr_bytes_read_per_rep[1]] + tkmap[ddr_bytes_write_per_rep[0]].dataframe[ddr_bytes_write_per_rep[1]]) / (tkmap[ddr_no_topdown_time_metric[0]].dataframe[ddr_no_topdown_time_metric[1]] / tkmap[ddr_reps[0]].dataframe[ddr_reps[1]])) / (10**9)
tkmap[hbm_mem_bw[0]].dataframe[hbm_mem_bw[1]] = ((tkmap[hbm_bytes_read_per_rep[0]].dataframe[hbm_bytes_read_per_rep[1]] + tkmap[hbm_bytes_write_per_rep[0]].dataframe[hbm_bytes_write_per_rep[1]]) / (tkmap[hbm_no_topdown_time_metric[0]].dataframe[hbm_no_topdown_time_metric[1]] / tkmap[hbm_reps[0]].dataframe[hbm_reps[1]])) / (10**9)
tkmap[mi250x_mem_bw[0]].dataframe[mi250x_mem_bw[1]] = ((tkmap[mi250x_bytes_read_per_rep[0]].dataframe[mi250x_bytes_read_per_rep[1]] + tkmap[mi250x_bytes_write_per_rep[0]].dataframe[mi250x_bytes_write_per_rep[1]]) / (tkmap[mi250x_time_metric[0]].dataframe[mi250x_time_metric[1]] / tkmap[mi250x_reps[0]].dataframe[mi250x_reps[1]])) / (10**9)
tkmap[v100_mem_bw[0]].dataframe[v100_mem_bw[1]] = ((tkmap[v100_bytes_read_per_rep[0]].dataframe[v100_bytes_read_per_rep[1]] + tkmap[v100_bytes_write_per_rep[0]].dataframe[v100_bytes_write_per_rep[1]]) / (tkmap[v100_time_metric[0]].dataframe[v100_time_metric[1]] / tkmap[v100_reps[0]].dataframe[v100_reps[1]])) / (10**9)
tkmap[mi300a_mem_bw[0]].dataframe[mi300a_mem_bw[1]] = ((tkmap[mi300a_bytes_read_per_rep[0]].dataframe[mi300a_bytes_read_per_rep[1]] + tkmap[mi300a_bytes_write_per_rep[0]].dataframe[mi300a_bytes_write_per_rep[1]]) / (tkmap[mi300a_time_metric[0]].dataframe[mi300a_time_metric[1]] / tkmap[mi300a_reps[0]].dataframe[mi300a_reps[1]])) / (10**9)
tkmap[a100_mem_bw[0]].dataframe[a100_mem_bw[1]] = ((tkmap[a100_bytes_read_per_rep[0]].dataframe[a100_bytes_read_per_rep[1]] + tkmap[a100_bytes_write_per_rep[0]].dataframe[a100_bytes_write_per_rep[1]]) / (tkmap[a100_time_metric[0]].dataframe[a100_time_metric[1]] / tkmap[a100_reps[0]].dataframe[a100_reps[1]])) / (10**9)
tkmap[gh200_mem_bw[0]].dataframe[gh200_mem_bw[1]] = ((tkmap[gh200_bytes_read_per_rep[0]].dataframe[gh200_bytes_read_per_rep[1]] + tkmap[gh200_bytes_write_per_rep[0]].dataframe[gh200_bytes_write_per_rep[1]]) / (tkmap[gh200_time_metric[0]].dataframe[gh200_time_metric[1]] / tkmap[gh200_reps[0]].dataframe[gh200_reps[1]])) / (10**9)

# Calculate atomic write bandwidth (in GB/sec)
tkmap[ddr_atom_bw[0]].dataframe[ddr_atom_bw[1]] = (tkmap[ddr_bytes_atomic_write_per_rep[0]].dataframe[ddr_bytes_atomic_write_per_rep[1]] / (tkmap[ddr_no_topdown_time_metric[0]].dataframe[ddr_no_topdown_time_metric[1]] / tkmap[ddr_reps[0]].dataframe[ddr_reps[1]])) / (10**9)
tkmap[hbm_atom_bw[0]].dataframe[hbm_atom_bw[1]] = (tkmap[hbm_bytes_atomic_write_per_rep[0]].dataframe[hbm_bytes_atomic_write_per_rep[1]] / (tkmap[hbm_no_topdown_time_metric[0]].dataframe[hbm_no_topdown_time_metric[1]] / tkmap[hbm_reps[0]].dataframe[hbm_reps[1]])) / (10**9)
tkmap[mi250x_atom_bw[0]].dataframe[mi250x_atom_bw[1]] = (tkmap[mi250x_bytes_atomic_write_per_rep[0]].dataframe[mi250x_bytes_atomic_write_per_rep[1]] / (tkmap[mi250x_time_metric[0]].dataframe[mi250x_time_metric[1]] / tkmap[mi250x_reps[0]].dataframe[mi250x_reps[1]])) / (10**9)
tkmap[v100_atom_bw[0]].dataframe[v100_atom_bw[1]] = (tkmap[v100_bytes_atomic_write_per_rep[0]].dataframe[v100_bytes_atomic_write_per_rep[1]] / (tkmap[v100_time_metric[0]].dataframe[v100_time_metric[1]] / tkmap[v100_reps[0]].dataframe[v100_reps[1]])) / (10**9)
tkmap[mi300a_atom_bw[0]].dataframe[mi300a_atom_bw[1]] = (tkmap[mi300a_bytes_atomic_write_per_rep[0]].dataframe[mi300a_bytes_atomic_write_per_rep[1]] / (tkmap[mi300a_time_metric[0]].dataframe[mi300a_time_metric[1]] / tkmap[mi300a_reps[0]].dataframe[mi300a_reps[1]])) / (10**9)
tkmap[a100_atom_bw[0]].dataframe[a100_atom_bw[1]] = (tkmap[a100_bytes_atomic_write_per_rep[0]].dataframe[a100_bytes_atomic_write_per_rep[1]] / (tkmap[a100_time_metric[0]].dataframe[a100_time_metric[1]] / tkmap[a100_reps[0]].dataframe[a100_reps[1]])) / (10**9)
tkmap[gh200_atom_bw[0]].dataframe[gh200_atom_bw[1]] = (tkmap[gh200_bytes_atomic_write_per_rep[0]].dataframe[gh200_bytes_atomic_write_per_rep[1]] / (tkmap[gh200_time_metric[0]].dataframe[gh200_time_metric[1]] / tkmap[gh200_reps[0]].dataframe[gh200_reps[1]])) / (10**9)

# Calculate unscaled bytes/rep
tkmap[ddr_bytes_per_rep[0]].dataframe[ddr_bytes_per_rep[1]] = (tkmap[ddr_bytes_read_per_rep[0]].dataframe[ddr_bytes_read_per_rep[1]] + tkmap[ddr_bytes_write_per_rep[0]].dataframe[ddr_bytes_write_per_rep[1]])
tkmap[hbm_bytes_per_rep[0]].dataframe[hbm_bytes_per_rep[1]] = (tkmap[hbm_bytes_read_per_rep[0]].dataframe[hbm_bytes_read_per_rep[1]] + tkmap[hbm_bytes_write_per_rep[0]].dataframe[hbm_bytes_write_per_rep[1]])
tkmap[mi250x_bytes_per_rep[0]].dataframe[mi250x_bytes_per_rep[1]] = (tkmap[mi250x_bytes_read_per_rep[0]].dataframe[mi250x_bytes_read_per_rep[1]] + tkmap[mi250x_bytes_write_per_rep[0]].dataframe[mi250x_bytes_write_per_rep[1]])
tkmap[v100_bytes_per_rep[0]].dataframe[v100_bytes_per_rep[1]] = (tkmap[v100_bytes_read_per_rep[0]].dataframe[v100_bytes_read_per_rep[1]] + tkmap[v100_bytes_write_per_rep[0]].dataframe[v100_bytes_write_per_rep[1]])
tkmap[mi300a_bytes_per_rep[0]].dataframe[mi300a_bytes_per_rep[1]] = (tkmap[mi300a_bytes_read_per_rep[0]].dataframe[mi300a_bytes_read_per_rep[1]] + tkmap[mi300a_bytes_write_per_rep[0]].dataframe[mi300a_bytes_write_per_rep[1]])
tkmap[a100_bytes_per_rep[0]].dataframe[a100_bytes_per_rep[1]] = (tkmap[a100_bytes_read_per_rep[0]].dataframe[a100_bytes_read_per_rep[1]] + tkmap[a100_bytes_write_per_rep[0]].dataframe[a100_bytes_write_per_rep[1]])
tkmap[gh200_bytes_per_rep[0]].dataframe[gh200_bytes_per_rep[1]] = (tkmap[gh200_bytes_read_per_rep[0]].dataframe[gh200_bytes_read_per_rep[1]] + tkmap[gh200_bytes_write_per_rep[0]].dataframe[gh200_bytes_write_per_rep[1]])


### 5.4 Define stats variables

This cell controls what columns are passed to:
* `thicket.stats.mean`
* `thicket.stats.minimum`
* `thicket.stats.maximum`
* `thicket.stats.std`

Users should set `stats_metrics_variable_names` to the names of the variables in the last two cells for which they
want these stats functions applied. The rest of the cells in this section will create new variables based on the following templates:
* `<input_variable_name>_mean`
* `<input_variable_name>_min`
* `<input_variable_name>_max`
* `<input_variable_name>_std`

In [11]:
# Names of all the variables defined above
stats_metrics_variable_names = [
    "ddr_no_topdown_time_metric",
    "hbm_no_topdown_time_metric",
    "mi250x_time_metric",
    "v100_time_metric",
    "mi300a_time_metric",
    "a100_time_metric",
    "gh200_time_metric",
    "ddr_bytes_read_per_rep",
    "hbm_bytes_read_per_rep",
    "mi250x_bytes_read_per_rep",
    "mi300a_bytes_read_per_rep",
    "a100_bytes_read_per_rep",
    "gh200_bytes_read_per_rep",
    "v100_bytes_read_per_rep",
    "ddr_bytes_write_per_rep",
    "hbm_bytes_write_per_rep",
    "mi250x_bytes_write_per_rep",
    "v100_bytes_write_per_rep",
    "mi300a_bytes_write_per_rep",
    "a100_bytes_write_per_rep",
    "gh200_bytes_write_per_rep",
    "ddr_bytes_read_per_rep_scaled",
    "hbm_bytes_read_per_rep_scaled",
    "mi250x_bytes_read_per_rep_scaled",
    "v100_bytes_read_per_rep_scaled",
    "mi300a_bytes_read_per_rep_scaled",
    "a100_bytes_read_per_rep_scaled",
    "gh200_bytes_read_per_rep_scaled",
    "ddr_bytes_write_per_rep_scaled",
    "hbm_bytes_write_per_rep_scaled",
    "mi250x_bytes_write_per_rep_scaled",
    "v100_bytes_write_per_rep_scaled",
    "mi300a_bytes_write_per_rep_scaled",
    "a100_bytes_write_per_rep_scaled",
    "gh200_bytes_write_per_rep_scaled",
    "ddr_flops_per_rep",
    "hbm_flops_per_rep",
    "mi250x_flops_per_rep",
    "v100_flops_per_rep",
    "mi300a_flops_per_rep",
    "a100_flops_per_rep",
    "gh200_flops_per_rep",
    "ddr_flops_per_rep_scaled",
    "hbm_flops_per_rep_scaled",
    "mi250x_flops_per_rep_scaled",
    "v100_flops_per_rep_scaled",
    "mi300a_flops_per_rep_scaled",
    "a100_flops_per_rep_scaled",
    "gh200_flops_per_rep_scaled",
    "ddr_bytes_per_rep",
    "hbm_bytes_per_rep",
    "mi250x_bytes_per_rep",
    "v100_bytes_per_rep",
    "mi300a_bytes_per_rep",
    "a100_bytes_per_rep",
    "gh200_bytes_per_rep",
    "ddr_bytes_per_rep_scaled",
    "hbm_bytes_per_rep_scaled",
    "mi250x_bytes_per_rep_scaled",
    "v100_bytes_per_rep_scaled",
    "mi300a_bytes_per_rep_scaled",
    "a100_bytes_per_rep_scaled",
    "gh200_bytes_per_rep_scaled",
    "ddr_flops_per_byte",
    "hbm_flops_per_byte",
    "mi250x_flops_per_byte",
    "v100_flops_per_byte",
    "mi300a_flops_per_byte",
    "a100_flops_per_byte",
    "gh200_flops_per_byte",
    "ddr_flop_rate",
    "hbm_flop_rate",
    "mi250x_flop_rate",
    "v100_flop_rate",
    "mi300a_flop_rate",
    "a100_flop_rate",
    "gh200_flop_rate",
    "ddr_read_bw",
    "hbm_read_bw",
    "mi250x_read_bw",
    "v100_read_bw",
    "mi300a_read_bw",
    "a100_read_bw",
    "gh200_read_bw",
    "ddr_write_bw",
    "hbm_write_bw",
    "mi250x_write_bw",
    "v100_write_bw",
    "mi300a_write_bw",
    "a100_write_bw",
    "gh200_write_bw",
    "ddr_mem_bw",
    "hbm_mem_bw",
    "mi250x_mem_bw",
    "v100_mem_bw",
    "mi300a_mem_bw",
    "a100_mem_bw",
    "gh200_mem_bw",
    "ddr_atom_bw",
    "hbm_atom_bw",
    "mi250x_atom_bw",
    "v100_atom_bw",
    "mi300a_atom_bw",
    "a100_atom_bw",
    "gh200_atom_bw",
    "ddr_tsteps",
    "hbm_tsteps",
    "v100_tsteps",
    "mi250x_tsteps",
    "mi300a_tsteps",
    "a100_tsteps",
    "gh200_tsteps",
    "ddr_be_bound",
    "ddr_mem_bound",
    "ddr_core_bound",
    "ddr_fe_bound",
    "ddr_fe_lat",
    "ddr_fe_bw",
    "ddr_bad_spec",
    "ddr_machine_clears",
    "ddr_retiring",
    "ddr_heavy_ops",
    "ddr_light_ops",
    "hbm_be_bound",
    "hbm_mem_bound",
    "hbm_core_bound",
    "hbm_fe_bound",
    "hbm_fe_lat",
    "hbm_fe_bw",
    "hbm_bad_spec",
    "hbm_br_mispred",
    "hbm_machine_clears",
    "hbm_retiring",
    "hbm_heavy_ops",
    "hbm_light_ops",
    "ddr_reps",
    "hbm_reps",
    "mi250x_reps",
    "mi300a_reps",
    "v100_reps",
    "a100_reps",
    "gh200_reps",
]

# Metrics to compute statistics for
exec("stats_metrics = [{}]".format(", ".join(stats_metrics_variable_names)))
# Match variable names to metrics
stats_met_map = zip(stats_metrics_variable_names, stats_metrics)

### 5.5 Compute statistics and add to python namespace

We use the `Thicket.stats.mean`, `Thicket.stats.minimum`, `Thicket.stats.maximum`, and `Thicket.stats.std` functions to calculate statistics for the metrics we defined above, on the 10 trials in our dataset. Also inject variable name into global namespace for usage below.

In [12]:
for var_name, col in stats_met_map:
    globals()[var_name + "_mean"] = (col[0], tt.stats.mean(tkmap[col[0]], columns=[col[1]])[0])
    globals()[var_name + "_min"] = (col[0], tt.stats.minimum(tkmap[col[0]], columns=[col[1]])[0])
    globals()[var_name + "_max"] = (col[0], tt.stats.maximum(tkmap[col[0]], columns=[col[1]])[0])
    globals()[var_name + "_std"] = (col[0], tt.stats.std(tkmap[col[0]], columns=[col[1]])[0])

### 5.6 Compose Thickets and add columns from statsframes

Quickly check all nodes are present, to ensure the rest of the notebook works correctly.

In [ ]:
def compare_string_lists(keys, lists):
    """
    Compare multiple lists of strings to check if they are the same.
    Report any strings present in one list but missing in others.

    Args:
        lists: Variable number of lists to compare.

    Returns:
        dict: A dictionary where keys are list indices and values are sets of unique strings
              present in that list but missing in others.
    """
    # Convert each list to a set
    sets = [set(lst) for lst in lists]

    # Find the union of all sets (all unique strings across all lists)
    all_strings = set.union(*sets)

    # Check for missing values in each list
    differences = {
        f"List {i+1}": all_strings - s
        for i, s in enumerate(sets)
    }

    # If all differences are empty, the lists are identical
    all_empty = all(not diff for diff in differences.values())
    if all_empty:
        return "All lists contain the same strings."

    print(differences)

    if len(differences) > 0:
        raise ValueError(f"All of {keys} must have the exact same kernels in the same order for the remainder of the notebook to work. Try excluding the kernels reported here and re-running.")
    
    return differences

compare_string_lists(tkmap.keys(), [list(tkmap[key].statsframe.dataframe["name"]) for key in tkmap.keys()])

In [14]:
th = tt.Thicket.concat_thickets(tkmap.values(), fill_perfdata=False)

for name in tkmap.keys():
    for col in tkmap[name].statsframe.dataframe:
        th.statsframe.dataframe[(name, col)] = tkmap[name].statsframe.dataframe[col]

### 5.7 Calculate speedup and metric ratios

In [15]:
ddr_to_hbm_speedup = ("hbm", "speedup")
ddr_to_mi250x_speedup = ("mi250x", "speedup")
ddr_to_mi300a_speedup = ("mi300a", "speedup")
ddr_to_v100_speedup = ("v100", "speedup")
ddr_to_a100_speedup = ("a100", "speedup")
ddr_to_gh200_speedup = ("gh200", "speedup")
ddr_hbm_mem_bound_ratio = ("Memory bound ratio", "")
ddr_hbm_retiring_ratio = ("Retiring ratio", "")
ddr_hbm_flop_rate_ratio = ("Flop rate ratio", "")

ddr_to_hbm_flops_speedup = ("hbm", "flops_speedup")
ddr_to_mi250x_flops_speedup = ("mi250x", "flops_speedup")
ddr_to_mi300a_flops_speedup = ("mi300a", "flops_speedup")
ddr_to_v100_flops_speedup = ("v100", "flops_speedup")
ddr_to_a100_flops_speedup = ("a100", "flops_speedup")
ddr_to_gh200_flops_speedup = ("gh200", "flops_speedup")

ddr_to_hbm_bandwidth_speedup = ("hbm", "bandwidth_speedup")
ddr_to_mi250x_bandwidth_speedup = ("mi250x", "bandwidth_speedup")
ddr_to_mi300a_bandwidth_speedup = ("mi300a", "bandwidth_speedup")
ddr_to_v100_bandwidth_speedup = ("v100", "bandwidth_speedup")
ddr_to_a100_bandwidth_speedup = ("a100", "bandwidth_speedup")
ddr_to_gh200_bandwidth_speedup = ("gh200", "bandwidth_speedup")

# Dividing by reps won't effect data if all hardware ran for same # reps
th.statsframe.dataframe[ddr_to_hbm_speedup] = tkmap[ddr_no_topdown_time_metric_mean[0]].statsframe.dataframe[ddr_no_topdown_time_metric_mean[1]] / tkmap[hbm_no_topdown_time_metric_mean[0]].statsframe.dataframe[hbm_no_topdown_time_metric_mean[1]] * tkmap[hbm_reps_mean[0]].statsframe.dataframe[hbm_reps_mean[1]] / tkmap[ddr_reps_mean[0]].statsframe.dataframe[ddr_reps_mean[1]]
th.statsframe.dataframe[ddr_to_mi250x_speedup] = tkmap[ddr_no_topdown_time_metric_mean[0]].statsframe.dataframe[ddr_no_topdown_time_metric_mean[1]] / tkmap[mi250x_time_metric_mean[0]].statsframe.dataframe[mi250x_time_metric_mean[1]] * tkmap[mi250x_reps_mean[0]].statsframe.dataframe[mi250x_reps_mean[1]] / tkmap[ddr_reps_mean[0]].statsframe.dataframe[ddr_reps_mean[1]]
th.statsframe.dataframe[ddr_to_mi300a_speedup] = tkmap[ddr_no_topdown_time_metric_mean[0]].statsframe.dataframe[ddr_no_topdown_time_metric_mean[1]] / tkmap[mi300a_time_metric_mean[0]].statsframe.dataframe[mi300a_time_metric_mean[1]] * tkmap[mi300a_reps_mean[0]].statsframe.dataframe[mi300a_reps_mean[1]] / tkmap[ddr_reps_mean[0]].statsframe.dataframe[ddr_reps_mean[1]]
th.statsframe.dataframe[ddr_to_v100_speedup] = tkmap[ddr_no_topdown_time_metric_mean[0]].statsframe.dataframe[ddr_no_topdown_time_metric_mean[1]] / tkmap[v100_time_metric_mean[0]].statsframe.dataframe[v100_time_metric_mean[1]] * tkmap[v100_reps_mean[0]].statsframe.dataframe[v100_reps_mean[1]] / tkmap[ddr_reps_mean[0]].statsframe.dataframe[ddr_reps_mean[1]]
th.statsframe.dataframe[ddr_to_a100_speedup] = tkmap[ddr_no_topdown_time_metric_mean[0]].statsframe.dataframe[ddr_no_topdown_time_metric_mean[1]] / tkmap[a100_time_metric_mean[0]].statsframe.dataframe[a100_time_metric_mean[1]] * tkmap[a100_reps_mean[0]].statsframe.dataframe[a100_reps_mean[1]] / tkmap[ddr_reps_mean[0]].statsframe.dataframe[ddr_reps_mean[1]]
th.statsframe.dataframe[ddr_to_gh200_speedup] = tkmap[ddr_no_topdown_time_metric_mean[0]].statsframe.dataframe[ddr_no_topdown_time_metric_mean[1]] / tkmap[gh200_time_metric_mean[0]].statsframe.dataframe[gh200_time_metric_mean[1]] * tkmap[gh200_reps_mean[0]].statsframe.dataframe[gh200_reps_mean[1]] / tkmap[ddr_reps_mean[0]].statsframe.dataframe[ddr_reps_mean[1]]

th.statsframe.dataframe[ddr_hbm_mem_bound_ratio] = th.statsframe.dataframe[ddr_mem_bound_mean] / th.statsframe.dataframe[hbm_mem_bound_mean]
th.statsframe.dataframe[ddr_hbm_retiring_ratio] = th.statsframe.dataframe[ddr_retiring_mean] / th.statsframe.dataframe[hbm_retiring_mean]
th.statsframe.dataframe[ddr_hbm_flop_rate_ratio] = th.statsframe.dataframe[ddr_flop_rate_mean] / th.statsframe.dataframe[hbm_flop_rate_mean]

# th.statsframe.dataframe[ddr_to_hbm_flops_speedup] = th.statsframe.dataframe[hbm_flop_rate_mean] / th.statsframe.dataframe[ddr_flop_rate_mean]
# th.statsframe.dataframe[ddr_to_mi250x_flops_speedup] = th.statsframe.dataframe[mi250x_flop_rate_mean] / th.statsframe.dataframe[ddr_flop_rate_mean]
# th.statsframe.dataframe[ddr_to_mi300a_flops_speedup] = th.statsframe.dataframe[mi300a_flop_rate_mean] / th.statsframe.dataframe[ddr_flop_rate_mean]
# th.statsframe.dataframe[ddr_to_v100_flops_speedup] = th.statsframe.dataframe[v100_flop_rate_mean] / th.statsframe.dataframe[ddr_flop_rate_mean]
# th.statsframe.dataframe[ddr_to_a100_flops_speedup] = th.statsframe.dataframe[a100_flop_rate_mean] / th.statsframe.dataframe[ddr_flop_rate_mean]
# th.statsframe.dataframe[ddr_to_gh200_flops_speedup] = th.statsframe.dataframe[gh200_flop_rate_mean] / th.statsframe.dataframe[ddr_flop_rate_mean]

# th.statsframe.dataframe[ddr_to_hbm_bandwidth_speedup] = th.statsframe.dataframe[hbm_mem_bw_mean] / th.statsframe.dataframe[ddr_mem_bw_mean]
# th.statsframe.dataframe[ddr_to_mi250x_bandwidth_speedup] = th.statsframe.dataframe[mi250x_mem_bw_mean] / th.statsframe.dataframe[ddr_mem_bw_mean]
# th.statsframe.dataframe[ddr_to_mi300a_bandwidth_speedup] = th.statsframe.dataframe[mi300a_mem_bw_mean] / th.statsframe.dataframe[ddr_mem_bw_mean]
# th.statsframe.dataframe[ddr_to_v100_bandwidth_speedup] = th.statsframe.dataframe[v100_mem_bw_mean] / th.statsframe.dataframe[ddr_mem_bw_mean]
# th.statsframe.dataframe[ddr_to_a100_bandwidth_speedup] = th.statsframe.dataframe[a100_mem_bw_mean] / th.statsframe.dataframe[ddr_mem_bw_mean]
# th.statsframe.dataframe[ddr_to_gh200_bandwidth_speedup] = th.statsframe.dataframe[gh200_mem_bw_mean] / th.statsframe.dataframe[ddr_mem_bw_mean]

In [ ]:
th.statsframe.dataframe[[hbm_mem_bw_mean, ddr_mem_bw_mean, a100_mem_bw_mean]]

## 6. Plotting

### 6.1 Flop Rate vs Memory Bandwidth scatter plots for DDR, HBM, V100, and MI250X colored by kernel type

In [17]:
def plot_straight_line(ax, p0, p1, **kwargs):
    _, xmax = ax.get_xlim()
    slope = (p1[1] - p0[1]) / (p1[0] - p0[0])
    y_intercept = slope * (0 - p1[0]) + p1[1]
    ymax = slope * xmax + y_intercept
    ax.plot([p0[0], xmax], [p0[1], ymax], **kwargs)

In [18]:
mem_v_flop_df = th.statsframe.dataframe[[
    "name",
    ddr_mem_bw_mean,
    ddr_flop_rate_mean,
    hbm_mem_bw_mean,
    hbm_flop_rate_mean,
    v100_mem_bw_mean,
    v100_flop_rate_mean,
    mi250x_mem_bw_mean,
    mi250x_flop_rate_mean,
    a100_mem_bw_mean,
    a100_flop_rate_mean,
    gh200_mem_bw_mean,
    gh200_flop_rate_mean,
    mi300a_mem_bw_mean,
    mi300a_flop_rate_mean,
]].copy(deep=True)
mem_v_flop_df.reset_index(inplace=True)
mem_v_flop_df[("groups", "")] = mem_v_flop_df["name"].str.split("_", n=1, expand=True).iloc[:, 0]
mem_v_flop_df[("ddr", "flop_gt_mem")] = mem_v_flop_df[ddr_flop_rate_mean] > mem_v_flop_df[ddr_mem_bw_mean]
mem_v_flop_df[("hbm", "flop_gt_mem")] = mem_v_flop_df[hbm_flop_rate_mean] > mem_v_flop_df[hbm_mem_bw_mean]
mem_v_flop_df[("v100", "flop_gt_mem")] = mem_v_flop_df[v100_flop_rate_mean] > mem_v_flop_df[v100_mem_bw_mean]
mem_v_flop_df[("mi250x", "flop_gt_mem")] = mem_v_flop_df[mi250x_flop_rate_mean] > mem_v_flop_df[mi250x_mem_bw_mean]
mem_v_flop_df[("a100", "flop_gt_mem")] = mem_v_flop_df[a100_flop_rate_mean] > mem_v_flop_df[a100_mem_bw_mean]
mem_v_flop_df[("gh200", "flop_gt_mem")] = mem_v_flop_df[gh200_flop_rate_mean] > mem_v_flop_df[gh200_mem_bw_mean]
mem_v_flop_df[("mi300a", "flop_gt_mem")] = mem_v_flop_df[mi300a_flop_rate_mean] > mem_v_flop_df[mi300a_mem_bw_mean]

In [ ]:
# DDR Plot
fig_mem_v_flop_ddr, ax_mem_v_flop_ddr = plt.subplots(figsize=(6, 6))
sns.scatterplot(
    x=mem_v_flop_df[ddr_mem_bw_mean],
    y=mem_v_flop_df[ddr_flop_rate_mean],
    hue=mem_v_flop_df[("groups", "")],
    ax=ax_mem_v_flop_ddr,
    s=4 * mpl.rcParams["lines.markersize"] ** 2,
)
ax_mem_v_flop_ddr.set_xlim(left=0, right=4000)
ax_mem_v_flop_ddr.set_ylim(bottom=0, top=10000)
ax_mem_v_flop_ddr.set_xticks([0, 1000, 2000, 3000, 4000], [0, 1000, 2000, 3000, 4000])
ax_mem_v_flop_ddr.set_xlabel("Memory Bandwidth (GB/sec)", fontsize=16)
ax_mem_v_flop_ddr.set_ylabel("Flop Rate (GFLOPS)", fontsize=16)
ax_mem_v_flop_ddr.tick_params(axis="both", which="major", labelsize=14)
plot_straight_line(ax_mem_v_flop_ddr, [0, 0], [mem_v_flop_df[ddr_mem_bw_mean].max(), mem_v_flop_df[ddr_mem_bw_mean].max()], c="k", linestyle="--", linewidth=2)
sns.move_legend(ax_mem_v_flop_ddr, "center", title="Category")

mem_v_flop_figlegend = plt.figure(figsize=(8,2))
mem_v_flop_patches, mem_v_flop_labels = ax_mem_v_flop_ddr.get_legend_handles_labels()
mem_v_flop_figlegend.legend(mem_v_flop_patches, mem_v_flop_labels, title="Kernel Group", ncols=7, fontsize=18, title_fontsize=18)
ax_mem_v_flop_ddr.get_legend().remove()

In [ ]:
# HBM Plot
fig_mem_v_flop_hbm, ax_mem_v_flop_hbm = plt.subplots(figsize=(6, 6))
sns.scatterplot(
    x=mem_v_flop_df[hbm_mem_bw_mean],
    y=mem_v_flop_df[hbm_flop_rate_mean],
    hue=mem_v_flop_df[("groups", "")],
    ax=ax_mem_v_flop_hbm,
    s=4 * mpl.rcParams["lines.markersize"] ** 2,
)
ax_mem_v_flop_hbm.set_xlim(left=0, right=4000)
ax_mem_v_flop_hbm.set_ylim(bottom=0, top=10000)
ax_mem_v_flop_hbm.set_xticks([0, 1000, 2000, 3000, 4000], [0, 1000, 2000, 3000, 4000])
ax_mem_v_flop_hbm.set_xlabel("Memory Bandwidth (GB/sec)", fontsize=16)
ax_mem_v_flop_hbm.set_ylabel("Flop Rate (GFLOPS)", fontsize=16)
ax_mem_v_flop_hbm.tick_params(axis="both", which="major", labelsize=14)
plot_straight_line(ax_mem_v_flop_hbm, [0, 0], [mem_v_flop_df[hbm_mem_bw_mean].max(), mem_v_flop_df[hbm_mem_bw_mean].max()], c="k", linestyle="--", linewidth=2)
ax_mem_v_flop_hbm.get_legend().remove()

In [ ]:
# V100 Plot
fig_mem_v_flop_v100, ax_mem_v_flop_v100 = plt.subplots(figsize=(6, 6))
sns.scatterplot(
    x=mem_v_flop_df[v100_mem_bw_mean],
    y=mem_v_flop_df[v100_flop_rate_mean],
    hue=mem_v_flop_df[("groups", "")],
    ax=ax_mem_v_flop_v100,
    s=4 * mpl.rcParams["lines.markersize"] ** 2,
)
ax_mem_v_flop_v100.set_xlim(left=0, right=4000)
ax_mem_v_flop_v100.set_ylim(bottom=0, top=10000)
ax_mem_v_flop_v100.set_xticks([0, 1000, 2000, 3000, 4000], [0, 1000, 2000, 3000, 4000])
ax_mem_v_flop_v100.set_xlabel("Memory Bandwidth (GB/sec)", fontsize=16)
ax_mem_v_flop_v100.set_ylabel("Flop Rate (GFLOPS)", fontsize=16)
ax_mem_v_flop_v100.tick_params(axis="both", which="major", labelsize=14)
plot_straight_line(ax_mem_v_flop_v100, [0, 0], [mem_v_flop_df[v100_mem_bw_mean].max(), mem_v_flop_df[v100_mem_bw_mean].max()], c="k", linestyle="--", linewidth=2)
ax_mem_v_flop_v100.get_legend().remove()

In [ ]:
# a100 Plot
fig_mem_v_flop_a100, ax_mem_v_flop_a100 = plt.subplots(figsize=(6, 6))
sns.scatterplot(
    x=mem_v_flop_df[a100_mem_bw_mean],
    y=mem_v_flop_df[a100_flop_rate_mean],
    hue=mem_v_flop_df[("groups", "")],
    ax=ax_mem_v_flop_a100,
    s=4 * mpl.rcParams["lines.markersize"] ** 2,
)
ax_mem_v_flop_a100.set_xlim(left=0, right=4000)
ax_mem_v_flop_a100.set_ylim(bottom=0, top=10000)
ax_mem_v_flop_a100.set_xticks([0, 1000, 2000, 3000, 4000], [0, 1000, 2000, 3000, 4000])
ax_mem_v_flop_a100.set_xlabel("Memory Bandwidth (GB/sec)", fontsize=16)
ax_mem_v_flop_a100.set_ylabel("Flop Rate (GFLOPS)", fontsize=16)
ax_mem_v_flop_a100.tick_params(axis="both", which="major", labelsize=14)
plot_straight_line(ax_mem_v_flop_a100, [0, 0], [mem_v_flop_df[a100_mem_bw_mean].max(), mem_v_flop_df[a100_mem_bw_mean].max()], c="k", linestyle="--", linewidth=2)
ax_mem_v_flop_a100.get_legend().remove()

In [ ]:
# MI250X Plot
fig_mem_v_flop_mi250x, ax_mem_v_flop_mi250x = plt.subplots(figsize=(6*(11000 / 3500), 6))

mi250x_ylim = 10000
mi250x_ylim_pad = 200

new_mi250x_x = mem_v_flop_df[mi250x_mem_bw_mean].tolist()
new_mi250x_y = mem_v_flop_df[mi250x_flop_rate_mean].tolist()
label_mi250x_markers = [0 for _ in range(len(new_mi250x_x))]
labels_mi250x = []
for i in range(len(new_mi250x_x)):
    y = new_mi250x_y[i]
    if y > mi250x_ylim:
        labels_mi250x.append((new_mi250x_x[i], new_mi250x_y[i], i))
        new_mi250x_y[i] = mi250x_ylim - mi250x_ylim_pad
        label_mi250x_markers[i] = 1
        
labels_mi250x = list(sorted(labels_mi250x, key=lambda tup: tup[0]))
label_mi250x_x = [tup[0] for tup in labels_mi250x]
label_mi250x_y = [tup[1] for tup in labels_mi250x]
label_mi250x_names = [tup[2] for tup in labels_mi250x]
        
label_mi250x_names = [mem_v_flop_df["name"].iloc[i] for i in label_mi250x_names]
print(label_mi250x_x)
print(label_mi250x_y)
print(label_mi250x_names)

sns.scatterplot(
    x=np.array(new_mi250x_x),  # mem_v_flop_df[mi250x_mem_bw_mean],
    y=np.array(new_mi250x_y),  # mem_v_flop_df[mi250x_flop_rate_mean],
    hue=mem_v_flop_df[("groups", "")],
    style=label_mi250x_markers,
    ax=ax_mem_v_flop_mi250x,
    s=4 * mpl.rcParams["lines.markersize"] ** 2,
)
ax_mem_v_flop_mi250x.set_xlim(left=0, right=11000)
ax_mem_v_flop_mi250x.set_ylim(bottom=0, top=mi250x_ylim) # 90000
ax_mem_v_flop_mi250x.set_xticks(list(range(0, 11001, 1000)), list(range(0, 11001, 1000)))
ax_mem_v_flop_mi250x.set_xlabel("Memory Bandwidth (GB/sec)", fontsize=16)
ax_mem_v_flop_mi250x.set_ylabel("Flop Rate (GFLOPS)", fontsize=16)
ax_mem_v_flop_mi250x.tick_params(axis="both", which="major", labelsize=14)
plot_straight_line(ax_mem_v_flop_mi250x, [0, 0], [mem_v_flop_df[mi250x_mem_bw_mean].max(), mem_v_flop_df[mi250x_mem_bw_mean].max()], c="k", linestyle="--", linewidth=2)

for i in range(len(label_mi250x_x)):
    label_pt = (label_mi250x_x[i], mi250x_ylim - 2.75*mi250x_ylim_pad)
    ax_mem_v_flop_mi250x.annotate(
        "{}".format(i),
        xy=label_pt,
        xytext=(0,0),
        fontsize=14,
        textcoords="offset points",
        ha="center",
        va="center",
    )

ax_mem_v_flop_mi250x.get_legend().remove()

## Performance Portability Metric

In [ ]:
# Architectural Efficiency Scores = Average FLOP or BW rate / peak theoretical

peak_gflops = {"mi250x": 191500, "mi300a": 248000, "v100": 31200, "a100": 19400, "gh200": 34000, "ddr": 4700, "hbm": 4700}
peak_bw_gb = {"mi250x": 12800, "mi300a": 21200, "v100": 3600, "a100": 3200, "gh200": 4000, "ddr": 600, "hbm": 3300}

mem_v_flop_df[("ddr", "arch_eff_gflops")] = mem_v_flop_df[ddr_flop_rate_mean] / peak_gflops["ddr"] * 100
mem_v_flop_df[("hbm", "arch_eff_gflops")] = mem_v_flop_df[hbm_flop_rate_mean] / peak_gflops["hbm"] * 100
mem_v_flop_df[("v100", "arch_eff_gflops")] = mem_v_flop_df[v100_flop_rate_mean] / peak_gflops["v100"] * 100
mem_v_flop_df[("a100", "arch_eff_gflops")] = mem_v_flop_df[a100_flop_rate_mean] / peak_gflops["a100"] * 100
mem_v_flop_df[("mi250x", "arch_eff_gflops")] = mem_v_flop_df[mi250x_flop_rate_mean] / peak_gflops["mi250x"] * 100
mem_v_flop_df[("gh200", "arch_eff_gflops")] = mem_v_flop_df[gh200_flop_rate_mean] / peak_gflops["gh200"] * 100
mem_v_flop_df[("mi300a", "arch_eff_gflops")] = mem_v_flop_df[mi300a_flop_rate_mean] / peak_gflops["mi300a"] * 100

mem_v_flop_df[("ddr", "arch_eff_bw_gb")] = mem_v_flop_df[ddr_mem_bw_mean] / peak_bw_gb["ddr"] * 100
mem_v_flop_df[("hbm", "arch_eff_bw_gb")] = mem_v_flop_df[hbm_mem_bw_mean] / peak_bw_gb["hbm"] * 100
mem_v_flop_df[("v100", "arch_eff_bw_gb")] = mem_v_flop_df[v100_mem_bw_mean] / peak_bw_gb["v100"] * 100
mem_v_flop_df[("a100", "arch_eff_bw_gb")] = mem_v_flop_df[a100_mem_bw_mean] / peak_bw_gb["a100"] * 100
mem_v_flop_df[("mi250x", "arch_eff_bw_gb")] = mem_v_flop_df[mi250x_mem_bw_mean] / peak_bw_gb["mi250x"] * 100
mem_v_flop_df[("gh200", "arch_eff_bw_gb")] = mem_v_flop_df[gh200_mem_bw_mean] / peak_bw_gb["gh200"] * 100
mem_v_flop_df[("mi300a", "arch_eff_bw_gb")] = mem_v_flop_df[mi300a_mem_bw_mean] / peak_bw_gb["mi300a"] * 100

mem_v_flop_df.head()

In [ ]:
mem_v_flop_df.reset_index()[mem_v_flop_df["name"] == "Basic_MAT_MAT_SHARED"][[
    ("ddr", "FLOP rate (GFLOPS)_mean"), ("ddr", "arch_eff_gflops"),
    ("hbm", "FLOP rate (GFLOPS)_mean"), ("hbm", "arch_eff_gflops"),
    ("v100", "FLOP rate (GFLOPS)_mean"), ("v100", "arch_eff_gflops"),
    ("a100", "FLOP rate (GFLOPS)_mean"), ("a100", "arch_eff_gflops"),
    ("gh200", "FLOP rate (GFLOPS)_mean"), ("gh200", "arch_eff_gflops"),
    ("mi250x", "FLOP rate (GFLOPS)_mean"), ("mi250x", "arch_eff_gflops"),
    ("mi300a", "FLOP rate (GFLOPS)_mean"), ("mi300a", "arch_eff_gflops"),
    ]]

In [ ]:
mem_v_flop_df.reset_index()[mem_v_flop_df["name"] == "Stream_TRIAD"][[
    ("ddr", "Memory Bandwidth (GB/sec)_mean"), ("ddr", "arch_eff_bw_gb"),
    ("hbm", "Memory Bandwidth (GB/sec)_mean"), ("hbm", "arch_eff_bw_gb"),
    ("v100", "Memory Bandwidth (GB/sec)_mean"), ("v100", "arch_eff_bw_gb"),
    ("a100", "Memory Bandwidth (GB/sec)_mean"), ("a100", "arch_eff_bw_gb"),
    ("gh200", "Memory Bandwidth (GB/sec)_mean"), ("gh200", "arch_eff_bw_gb"),
    ("mi250x", "Memory Bandwidth (GB/sec)_mean"), ("mi250x", "arch_eff_bw_gb"),
    ("mi300a", "Memory Bandwidth (GB/sec)_mean"), ("mi300a", "arch_eff_bw_gb"),
]]

In [ ]:
mem_v_flop_df[[
    "name",
    ("ddr", "Memory Bandwidth (GB/sec)_mean"), ("ddr", "arch_eff_bw_gb"),
    ("hbm", "Memory Bandwidth (GB/sec)_mean"), ("hbm", "arch_eff_bw_gb"),
    ("v100", "Memory Bandwidth (GB/sec)_mean"), ("v100", "arch_eff_bw_gb"),
    ("a100", "Memory Bandwidth (GB/sec)_mean"), ("a100", "arch_eff_bw_gb"),
    ("gh200", "Memory Bandwidth (GB/sec)_mean"), ("gh200", "arch_eff_bw_gb"),
    ("mi250x", "Memory Bandwidth (GB/sec)_mean"), ("mi250x", "arch_eff_bw_gb"),
    ("mi300a", "Memory Bandwidth (GB/sec)_mean"), ("mi300a", "arch_eff_bw_gb"),
]]

In [28]:
assert len(mem_v_flop_df) == 46

In [ ]:
arch_eff_df = mem_v_flop_df[['name', ("ddr", "arch_eff_gflops"), ("hbm", "arch_eff_gflops"), ("v100", "arch_eff_gflops"), ("a100", "arch_eff_gflops"), ("mi250x", "arch_eff_gflops"), ("gh200", "arch_eff_gflops"),
                            ("mi300a", "arch_eff_gflops"), ("ddr", "arch_eff_bw_gb"), ("hbm", "arch_eff_bw_gb"), ("v100", "arch_eff_bw_gb"), ("a100", "arch_eff_bw_gb"), ("mi250x", "arch_eff_bw_gb"), ("gh200", "arch_eff_bw_gb"), ("mi300a", "arch_eff_bw_gb") ]]
arch_eff_df.set_index("name", inplace=True)
sns.set_theme(rc={'figure.figsize':(20,20)})
arch_eff_heatmap = sns.heatmap(arch_eff_df, annot=True, vmin=0, vmax=100, cbar_kws={"shrink": 0.5}).get_figure()
arch_eff_heatmap.tight_layout()
arch_eff_heatmap.savefig("arch_eff_heatmap.svg") 
arch_eff_heatmap.show()

In [30]:
for name in list(arch_eff_df.reset_index()["name"]):
    if name not in Memory_bound+retiring_bound:
        print(name)

In [ ]:
# Harmonic Mean

for index, row in mem_v_flop_df.iterrows():
    #Flops-based
    mem_v_flop_df.at[index,'CPU FLOPS'] = stats.hmean([row[("ddr", "arch_eff_gflops")], row[("hbm", "arch_eff_gflops")],])
    mem_v_flop_df.at[index,'GPU-PREV FLOPs'] = stats.hmean([row[("v100", "arch_eff_gflops")], row[("a100", "arch_eff_gflops")], row[("mi250x", "arch_eff_gflops")],]) 
    mem_v_flop_df.at[index,'GPU-NEW FLOPs'] = stats.hmean([row[("gh200", "arch_eff_gflops")], row[("mi300a", "arch_eff_gflops")],]) 
    mem_v_flop_df.at[index,'GPU-NV FLOPs'] = stats.hmean([row[("v100", "arch_eff_gflops")], row[("a100", "arch_eff_gflops")], row[("gh200", "arch_eff_gflops")],]) 
    mem_v_flop_df.at[index,'GPU-AMD FLOPs'] = stats.hmean([row[("mi250x", "arch_eff_gflops")], row[("mi300a", "arch_eff_gflops")],]) 
    mem_v_flop_df.at[index,'GPU FLOPs'] = stats.hmean([row[("v100", "arch_eff_gflops")], row[("a100", "arch_eff_gflops")], 
                                                                row[("mi250x", "arch_eff_gflops")], row[("gh200", "arch_eff_gflops")], row[("mi300a", "arch_eff_gflops")],]) 
    mem_v_flop_df.at[index,'All FLOPs'] = stats.hmean([row[("ddr", "arch_eff_gflops")], row[("hbm", "arch_eff_gflops")], row[("v100", "arch_eff_gflops")], row[("a100", "arch_eff_gflops")], 
                                                                row[("mi250x", "arch_eff_gflops")], row[("gh200", "arch_eff_gflops")], row[("mi300a", "arch_eff_gflops")],])
    # Bandwidth-based
    mem_v_flop_df.at[index,'CPU BW'] = stats.hmean([row[("ddr", "arch_eff_bw_gb")], row[("hbm", "arch_eff_bw_gb")], ])
    mem_v_flop_df.at[index,'GPU-PREV BW'] = stats.hmean([row[("v100", "arch_eff_bw_gb")], row[("a100", "arch_eff_bw_gb")], row[("mi250x", "arch_eff_bw_gb")],])
    mem_v_flop_df.at[index,'GPU-NEW BW'] = stats.hmean([row[("gh200", "arch_eff_bw_gb")], row[("mi300a", "arch_eff_bw_gb")],])
    mem_v_flop_df.at[index,'GPU-NV BW'] = stats.hmean([row[("v100", "arch_eff_bw_gb")], row[("a100", "arch_eff_bw_gb")], row[("gh200", "arch_eff_bw_gb")],])
    mem_v_flop_df.at[index,'GPU-AMD BW'] = stats.hmean([row[("mi250x", "arch_eff_bw_gb")], row[("mi300a", "arch_eff_bw_gb")],])
    mem_v_flop_df.at[index,'GPU BW'] = stats.hmean([row[("v100", "arch_eff_bw_gb")], row[("a100", "arch_eff_bw_gb")], 
                                                                row[("mi250x", "arch_eff_bw_gb")], row[("gh200", "arch_eff_bw_gb")], row[("mi300a", "arch_eff_bw_gb")],])
    mem_v_flop_df.at[index,'All BW'] = stats.hmean([row[("ddr", "arch_eff_bw_gb")], row[("hbm", "arch_eff_bw_gb")], row[("v100", "arch_eff_bw_gb")], row[("a100", "arch_eff_bw_gb")], 
                                                                row[("mi250x", "arch_eff_bw_gb")], row[("gh200", "arch_eff_bw_gb")], row[("mi300a", "arch_eff_bw_gb")],])
    
perf_port_df = mem_v_flop_df[['name', 'CPU FLOPS', 'GPU-PREV FLOPs', 'GPU-NEW FLOPs','GPU-NV FLOPs', 'GPU-AMD FLOPs', 'GPU FLOPs', 'All FLOPs', 
                              'CPU BW', 'GPU-PREV BW', 'GPU-NEW BW', 'GPU-NV BW', 'GPU-AMD BW', 'GPU BW','All BW', ]]
perf_port_df.set_index("name", inplace=True)
perf_port_df.head()

In [ ]:
sns.set_theme(rc={'figure.figsize':(20,20)})
perf_port_heatmap = sns.heatmap(perf_port_df, annot=True, vmin=0, vmax=100, cbar_kws={"shrink": 0.5}).get_figure()
perf_port_heatmap.tight_layout()
perf_port_heatmap.savefig("perf_port_heatmap.svg") 
perf_port_heatmap.show()

### 6.2 Metric bar charts for each kernel

In [ ]:
#analytic_metric_color_map = sns.color_palette("colorblind")

analytic_met_read_df = th.statsframe.dataframe[[
    "name",
    ddr_bytes_read_per_rep_scaled_mean,
    ddr_flops_per_rep_scaled_mean,
    ddr_flops_per_byte_mean,
]]
analytic_met_write_df = th.statsframe.dataframe[[
    "name",
    ddr_bytes_write_per_rep_scaled_mean,
    ddr_flops_per_rep_scaled_mean,
    ddr_flops_per_byte_mean,
]]
analytic_met_read_df.columns = analytic_met_read_df.columns.to_flat_index()
analytic_met_read_df.rename(
    columns={
        "name": "Kernel",
        ddr_bytes_read_per_rep_scaled_mean: "BytesR/W",
        ddr_flops_per_rep_scaled_mean: "FLOPs",
        ddr_flops_per_byte_mean: "FLOPs/Byte",
    },
    inplace=True
)
analytic_met_read_df["FLOPs/Byte"] = analytic_met_read_df["FLOPs/Byte"].apply(lambda x: x if x < 1000 else 0.0)
analytic_met_write_df.columns = analytic_met_write_df.columns.to_flat_index()
analytic_met_write_df.rename(
    columns={
        "name": "Kernel",
        ddr_bytes_write_per_rep_scaled_mean: "BytesR/W",
        ddr_flops_per_rep_scaled_mean: "FLOPs",
        ddr_flops_per_byte_mean: "FLOPs/Byte",
    },
    inplace=True
)
analytic_met_read_df["type"] = "Read"
analytic_met_write_df["type"] = "Write"
analytic_met_df = pd.concat([analytic_met_read_df, analytic_met_write_df]).reset_index(drop=True)

analytic_met_read_df["color"] = 1

pg_analytic = sns.PairGrid(
    data=analytic_met_read_df,
    y_vars=["Kernel"],
    x_vars=["BytesR/W", "FLOPs", "FLOPs/Byte"],
    despine=True,
    height=10.5,
    aspect=0.2,
    hue="color"
)
pg_analytic.tight_layout()
pg_analytic.map(sns.barplot)

pg_analytic.figure.axes[0].clear()
sns.barplot(
    data=analytic_met_df,
    y="Kernel",
    x="BytesR/W",
    hue="type",
    ax=pg_analytic.figure.axes[0],
)
pg_analytic.figure.axes[0].get_yaxis().get_label().set_visible(False)

pg_analytic_y_ax = pg_analytic.figure.axes[0].get_yaxis()
pg_analytic_y_ax.set_tick_params(pad=160)
for tick in pg_analytic_y_ax.get_majorticklabels():
    tick.set_horizontalalignment("left")

In [ ]:
ddr_tma_df = th.statsframe.dataframe[[
    "name",
    ddr_fe_bound_mean,
    ddr_bad_spec_mean,
    ddr_retiring_mean,
    ddr_core_bound_mean,
    ddr_mem_bound_mean
]]
ddr_tma_df.columns = ddr_tma_df.columns.to_flat_index()
ddr_tma_df.rename(
    columns={
        "name": "Kernel",
        ddr_fe_bound_mean: "Frontend Bound",
        ddr_bad_spec_mean: "Bad Speculation",
        ddr_retiring_mean: "Retiring",
        ddr_core_bound_mean: "Core Bound",
        ddr_mem_bound_mean: "Memory Bound",
    },
    inplace=True
)

ddr_tma_df["color"] = 1

pg_ddr_tma = sns.PairGrid(
    data=ddr_tma_df,
    y_vars=["Kernel"],
    x_vars=[
        "Frontend Bound",
        "Bad Speculation",
        "Retiring",
        "Core Bound",
        "Memory Bound"
    ],
    despine=True,
    height=10.5,
    aspect=0.1,
    hue="color"
)
pg_ddr_tma.tight_layout()
pg_ddr_tma.map(sns.barplot)
pg_ddr_tma.figure.axes[0].get_yaxis().get_label().set_visible(False)
for ax in pg_ddr_tma.figure.axes:
    ax.set_xlim(0, 1)
    ax.set_xlabel(ax.get_xlabel().replace(" ", "\n"))
pg_ddr_tma_y_ax = pg_ddr_tma.figure.axes[0].get_yaxis()
pg_ddr_tma_y_ax.set_tick_params(pad=152)
for tick in pg_ddr_tma_y_ax.get_majorticklabels():
    tick.set_horizontalalignment("left")
#pg_ddr_tma.figure.savefig("figures/ddr_topdown_per_kernel.png", bbox_inches="tight", dpi=200)

In [ ]:
hbm_tma_df = th.statsframe.dataframe[[
    "name",
    # hbm_mem_bw_mean,
    # hbm_flop_rate_mean,
    hbm_fe_bound_mean,
    hbm_bad_spec_mean,
    hbm_retiring_mean,
    hbm_core_bound_mean,
    hbm_mem_bound_mean
]]
hbm_tma_df.columns = hbm_tma_df.columns.to_flat_index()
hbm_tma_df.rename(
    columns={
        "name": "Kernel",
        # hbm_mem_bw_mean: "Memory Bandwidth (GiB/s)",
        # hbm_flop_rate_mean: "FLOP Rate (GiFLOPS/s)",
        hbm_fe_bound_mean: "Frontend Bound",
        hbm_bad_spec_mean: "Bad Speculation",
        hbm_retiring_mean: "Retiring",
        hbm_core_bound_mean: "Core Bound",
        hbm_mem_bound_mean: "Memory Bound",
    },
    inplace=True
)

hbm_tma_df["color"] = 1

pg_hbm_tma = sns.PairGrid(
    data=hbm_tma_df,
    y_vars=["Kernel"],
    x_vars=[
        # "Memory Bandwidth (GiB/s)",
        # "FLOP Rate (GiFLOPS/s)",
        "Frontend Bound",
        "Bad Speculation",
        "Retiring",
        "Core Bound",
        "Memory Bound"
    ],
    despine=True,
    height=10.5,
    aspect=0.1,
    hue="color"
)
pg_hbm_tma.tight_layout()
pg_hbm_tma.map(sns.barplot)
pg_hbm_tma.figure.axes[0].get_yaxis().get_label().set_visible(False)
for ax in pg_hbm_tma.figure.axes:
    ax.set_xlim(0, 1)
    ax.set_xlabel(ax.get_xlabel().replace(" ", "\n"))
pg_hbm_tma_y_ax = pg_hbm_tma.figure.axes[0].get_yaxis()
pg_hbm_tma_y_ax.set_tick_params(pad=152)
for tick in pg_hbm_tma_y_ax.get_majorticklabels():
    tick.set_horizontalalignment("left")
#pg_hbm_tma.figure.savefig("figures/hbm_topdown_per_kernel.png", bbox_inches="tight", dpi=200)

# Speedup over SPR-DDR Plot

In [ ]:

triple_plot_df = th.statsframe.dataframe[["name", ddr_mem_bound_mean, ddr_to_hbm_speedup, ddr_to_v100_speedup, ddr_to_a100_speedup, ddr_to_gh200_speedup, ddr_to_mi250x_speedup, ddr_to_mi300a_speedup]].reset_index(drop=True)
triple_plot_df.reset_index(inplace=True)

triple_plot_df.columns = triple_plot_df.columns.to_flat_index()
triple_plot_df.rename(
    columns={
        "name": "Kernel",
        ddr_mem_bound_mean: "DDR Memory Bound",
        ddr_to_hbm_speedup: "SPR-HBM Speedup",
        ddr_to_v100_speedup: "V100 Speedup",
        ddr_to_a100_speedup: "A100 Speedup",
        ddr_to_gh200_speedup: "GH200 Speedup",
        ddr_to_mi250x_speedup: "MI250X Speedup",
        ddr_to_mi300a_speedup: "MI300A Speedup",
    },
    inplace=True
)

ddr_stream_copy_mem_bound = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["DDR Memory Bound"].iloc[0]
hbm_stream_copy_spdup = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["SPR-HBM Speedup"].iloc[0]
v100_stream_copy_spdup = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["V100 Speedup"].iloc[0]
mi250x_stream_copy_spdup = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["MI250X Speedup"].iloc[0]
mi300a_stream_copy_spdup = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["MI300A Speedup"].iloc[0]
a100_stream_copy_spdup = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["A100 Speedup"].iloc[0]
gh200_stream_copy_spdup = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["GH200 Speedup"].iloc[0]

sns.set_theme(palette="tab10", font_scale=0.8)
sns.set_style("ticks")
triple_plot_df["color"] = 1

pg_triple = sns.PairGrid(
    data=triple_plot_df,
    y_vars=["Kernel"],
    x_vars=["DDR Memory Bound", "SPR-HBM Speedup", "V100 Speedup", "A100 Speedup", "GH200 Speedup", "MI250X Speedup", "MI300A Speedup"],
    despine=True,
    height=10.5,
    aspect=0.15,
    hue="color"
)
pg_triple.tight_layout()
pg_triple.map(sns.barplot)
pg_triple.figure.axes[0].set_xlim(0, 1)
pg_triple.figure.axes[0].set_xlabel("SPR-DDR\nMemory Bound")
pg_triple.figure.axes[0].get_yaxis().get_label().set_visible(False)
pg_triple.figure.axes[0].axvline(ddr_stream_copy_mem_bound, color="gold")
pg_triple.figure.axes[1].axvline(1.0, color="red")
pg_triple.figure.axes[1].axvline(hbm_stream_copy_spdup, color="gold")
pg_triple.figure.axes[1].set_xlim(0, 40)
pg_triple.figure.axes[1].set_xlabel("SPR-HBM\nSpeedup")
pg_triple.figure.axes[2].axvline(1.0, color="red")
pg_triple.figure.axes[2].axvline(v100_stream_copy_spdup, color="gold")
pg_triple.figure.axes[2].set_xlim(0, 40)
pg_triple.figure.axes[2].set_xlabel("P9-V100\nSpeedup")
pg_triple.figure.axes[3].axvline(1.0, color="red")
pg_triple.figure.axes[3].axvline(a100_stream_copy_spdup, color="gold")
pg_triple.figure.axes[3].set_xlim(0, 40)
pg_triple.figure.axes[3].set_xlabel("CAS-A100\nSpeedup")
pg_triple.figure.axes[4].axvline(1.0, color="red")
pg_triple.figure.axes[4].axvline(gh200_stream_copy_spdup, color="gold")
pg_triple.figure.axes[4].set_xlim(0, 40)
pg_triple.figure.axes[4].set_xlabel("GR-GH200\nSpeedup")
pg_triple.figure.axes[5].axvline(1.0, color="red")
pg_triple.figure.axes[5].axvline(mi250x_stream_copy_spdup, color="gold")
pg_triple.figure.axes[5].set_xlim(0, 40)
pg_triple.figure.axes[5].set_xlabel("EPYC-MI250X\nSpeedup")
pg_triple.figure.axes[6].axvline(1.0, color="red")
pg_triple.figure.axes[6].axvline(mi300a_stream_copy_spdup, color="gold")
pg_triple.figure.axes[6].set_xlim(0, 40)
pg_triple.figure.axes[6].set_xlabel("EPYC-MI300A\nSpeedup")

gpu_stream_copy_spdups = {
    "V100": v100_stream_copy_spdup,
    "A100": a100_stream_copy_spdup,
    "GH200": gh200_stream_copy_spdup,
    "MI250X": mi250x_stream_copy_spdup,
    "MI300A": mi300a_stream_copy_spdup,
}

starting_subplot_idx = 2
for i, gpu_type in enumerate(["V100", "A100", "GH200", "MI250X", "MI300A"]):
    for label in pg_triple.figure.axes[0].get_yticklabels():
        _, label_y = label.get_position()
        min_x, max_x = pg_triple.figure.axes[starting_subplot_idx+i].get_xlim()
        curr_kernel_speedup = triple_plot_df[triple_plot_df["Kernel"] == label.get_text()]["{} Speedup".format(gpu_type)].iloc[0]
        if curr_kernel_speedup > max_x:
            pg_triple.figure.axes[starting_subplot_idx+i].annotate(
                "{:.2f}".format(curr_kernel_speedup),
                xy=(max_x, label_y),
                xytext=(2, 0),
                textcoords="offset points",
                ha="left",
                va="center",
            )
starting_subplot_idx = 1         
for label in pg_triple.figure.axes[0].get_yticklabels():
    _, label_y = label.get_position()
    curr_kernel_speedup = triple_plot_df[triple_plot_df["Kernel"] == label.get_text()]["SPR-HBM Speedup"].iloc[0]
    if curr_kernel_speedup > 1:
        pg_triple.figure.axes[starting_subplot_idx].annotate(
            "{:.2f}".format(curr_kernel_speedup),
            xy=(2, label_y),
            xytext=(3, 0),
            textcoords="offset points",
            ha="left",
            va="center",
        )

pg_triple_y_ax = pg_triple.figure.axes[0].get_yaxis()
pg_triple_y_ax.set_tick_params(pad=152)
for tick in pg_triple_y_ax.get_majorticklabels():
    tick.set_horizontalalignment("left")

### 6.3 Analysis of kernel performance using clustering

#### 6.3A Agglomerative hierarchical clustering

In [37]:
# Try bytes/rep, flops/rep, and flops/bytes (divide by problem size first)
clustering_metrics = [
    ddr_retiring_mean,
    ddr_fe_bound_mean,
    ddr_bad_spec_mean,
    ddr_core_bound_mean,
    ddr_mem_bound_mean,
]
clustering_data = th.statsframe.dataframe.loc[:, clustering_metrics].copy(deep=True)

In [38]:
# Determine outliers based on speedup
speedup_cols = [ddr_to_mi250x_speedup, ddr_to_v100_speedup, ddr_to_hbm_speedup]

for spdp in speedup_cols:
    # calculate z score for that column
    z = np.abs(stats.zscore(th.statsframe.dataframe[spdp]))
    # 3 is the typical z score threshold 
    outlier_indices = np.where(z > 3)[0]
    # if there is at least one outlier print the name
    if len(outlier_indices) > 0:    
        print(th.statsframe.dataframe.iloc[[outlier_indices[0]]].name.unique())

In [39]:
# We need to filter non O(n) complexity kernels (listed in the table 1 in the paper) as well as outliers based on speedup identified in cell above
Non_On_kernels = [
    "Algorithm_SORT",
    "Algorithm_SORTPAIRS",
    "Comm_HALO_EXCHANGE",
    "Comm_HALO_EXCHANGE_FUSED",
    "Comm_HALO_PACKING",
    "Comm_HALO_PACKING_FUSED",
    "Comm_HALO_SENDRECV",
    "Polybench_2MM",
    "Polybench_3MM" ,
    "Polybench_FLOYD_WARSHALL",
    "Polybench_GEMM",
    "Apps_EDGE3D",
    "Basic_PI_REDUCE",
]

clustering_data["name"] = [n.frame["name"] for n in clustering_data.index]
clustering_data = clustering_data[~clustering_data["name"].isin(Non_On_kernels)]
clustering_data = clustering_data.drop(columns=["name"])
# Filter from statsframe
th.statsframe.dataframe = th.statsframe.dataframe[~th.statsframe.dataframe["name"].isin(Non_On_kernels)]

In [ ]:
clustering_data

In [ ]:
thresh = 1.4
cluster_op = sk.AgglomerativeClustering(linkage="ward", distance_threshold=thresh, n_clusters=None, compute_full_tree=True)
th.statsframe.dataframe[("label", "")] = cluster_op.fit_predict(clustering_data)

cluster_summary_df = th.statsframe.dataframe[["name", *clustering_metrics, ddr_to_hbm_speedup, ddr_to_v100_speedup, ddr_to_mi250x_speedup, ("label", "")]].reset_index(drop=True)
cluster_summary_df.columns = cluster_summary_df.columns.to_flat_index()
cluster_summary_df.rename(
    columns={
        "name": "Kernel",
        ddr_fe_bound_mean: "Frontend Bound",
        ddr_bad_spec_mean: "Bad Speculation",
        ddr_retiring_mean: "Retiring",
        ddr_core_bound_mean: "Core Bound",
        ddr_mem_bound_mean: "Memory Bound",
        ddr_to_hbm_speedup: "Speedup on SPR-HBM",
        ddr_to_v100_speedup: "Speedup on P9-V100",
        ddr_to_mi250x_speedup: "Speedup on EPYC-MI250X",
        ("label", ""): "Cluster ID",
    },
    inplace=True
)
cluster_summary_df

#### 6.3B Dendrogram Visualization of Clusters

In [42]:
def plot_dendrogram(model, **kwargs):
    # Create linkage matrix and then plot the dendrogram

    # create the counts of samples under each node
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  # leaf node
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    linkage_matrix = np.column_stack(
        [model.children_, model.distances_, counts]
    ).astype(float)

    # Plot the corresponding dendrogram
    return dendrogram(linkage_matrix, **kwargs)

In [ ]:
dendro_fig, dendro_ax = plt.subplots(figsize=(25, 10))

with plt.rc_context({"lines.linewidth": 2}):
    dendro_dict = plot_dendrogram(
        cluster_op,
        ax=dendro_ax,
        color_threshold=thresh,
        labels=th.statsframe.dataframe["name"],
        above_threshold_color="darkgray",
    )

    dendro_colors = {th.statsframe.dataframe[("label", "")].iloc[l]: c for l, c in zip(dendro_dict["leaves"], dendro_dict["leaves_color_list"])}
    dendro_labels = list(sorted([l for l in dendro_colors.keys()]))
    dendro_handles = []
    for label in dendro_labels:
        dendro_handles.append(plt.Line2D([0], [0], color=dendro_colors[label]))

    dendro_fig.legend(
        dendro_handles,
        dendro_labels,
        title="Cluster",
        loc="upper right",
        bbox_to_anchor=(0.9, 0.9),
        title_fontsize=18,
        fontsize=18,
    )
    dendro_ax.axhline(thresh, color="k", linestyle="--")

dendro_ax.spines[["right", "top"]].set_visible(False)
dendro_ax.set_ylabel("Euclidean Distance between Cluster Centers", fontsize=20)
dendro_ax.set_xlabel("Kernels", fontsize=20)
dendro_ax.tick_params(axis="both", which="major", labelsize=18)

#dendro_fig.savefig("figures/clustering_dendrogram.png", bbox_inches="tight", dpi=200)

#### 6.3C Categorical plot showing the makeup of each cluster

In [44]:
tma_metric_dfs = []
for m in ["Retiring", "Frontend Bound", "Bad Speculation", "Core Bound", "Memory Bound"]:
    single_tma_metric_df = cluster_summary_df[["Kernel", m, "Cluster ID"]]
    single_tma_metric_df.rename(columns={m: "TMA Metric"}, inplace=True)
    single_tma_metric_df["type"] = m
    tma_metric_dfs.append(single_tma_metric_df)

clustering_plotting_metric_df = pd.concat(tma_metric_dfs)

In [ ]:
g = sns.catplot(
    data=clustering_plotting_metric_df,
    x="Cluster ID",
    y="TMA Metric",
    hue="type",
    kind="bar",
    palette="colorblind",
)
g.set_axis_labels("", "% Runtime")
g.set(ylim=(0, 1))
g.set_xticklabels(["Compute Bound Cluster", "Memory Bound Cluster"])
g.legend.set_title("Topdown Metrics")

g.savefig("new_top_metrics_o_n.png")


In [ ]:
# Grab just the relevant metrics from above  and flatten for parallel coordinate plots
relevant_metrics_df = th.statsframe.dataframe[[
    "name",
    ddr_fe_bound_mean,
    ddr_bad_spec_mean,
    ddr_retiring_mean,
    ddr_core_bound_mean,
    ddr_mem_bound_mean,
    ddr_to_hbm_speedup,
    ddr_to_v100_speedup,
    ddr_to_a100_speedup,
    ddr_to_gh200_speedup,
    ddr_to_mi250x_speedup,
    ddr_to_mi300a_speedup,
    ("label", ""),
]].copy(deep=True)
relevant_metrics_df.columns = relevant_metrics_df.columns.to_flat_index()
relevant_metrics_df.rename(
    columns={
        "name": "Kernel",
        ddr_fe_bound_mean: "Frontend Bound",
        ddr_bad_spec_mean: "Bad Speculation",
        ddr_retiring_mean: "Retiring",
        ddr_core_bound_mean: "Core Bound",
        ddr_mem_bound_mean: "Memory Bound",
        ddr_to_hbm_speedup: "Speedup on SPR-HBM",
        ddr_to_v100_speedup: "Speedup on P9-V100",
        ddr_to_a100_speedup: "Speedup on CAS-A100",
        ddr_to_gh200_speedup: "Speedup on GR-GH200",
        ddr_to_mi250x_speedup: "Speedup on EPYC-MI250X",
        ddr_to_mi300a_speedup: "Speedup on MI300A",
        ("label", ""): "Cluster ID",
    },
    inplace=True,
)
relevant_metrics_df.reset_index(inplace=True, drop=True)
relevant_metrics_df.sort_values(by="Kernel", inplace=True)
relevant_metrics_df

In [ ]:
# 0 purple, 1 green, 2 red, 3 orange
triple_plot_df = th.statsframe.dataframe[["name", ddr_mem_bound_mean, ddr_to_v100_speedup, ddr_to_a100_speedup, ddr_to_gh200_speedup, ddr_to_mi250x_speedup, ddr_to_mi300a_speedup]].reset_index(drop=True)
triple_plot_df.reset_index(inplace=True)

triple_plot_df.columns = triple_plot_df.columns.to_flat_index()
triple_plot_df.rename(
    columns={
        "name": "Kernel",
        ddr_mem_bound_mean: "DDR Memory Bound",
        ddr_to_v100_speedup: "V100 Speedup",
        ddr_to_a100_speedup: "A100 Speedup",
        ddr_to_gh200_speedup: "GH200 Speedup",
        ddr_to_mi250x_speedup: "MI250X Speedup",
        ddr_to_mi300a_speedup: "MI300A Speedup",
    },
    inplace=True
)

ddr_stream_copy_mem_bound = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["DDR Memory Bound"].iloc[0]
v100_stream_copy_spdup = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["V100 Speedup"].iloc[0]
a100_stream_copy_spdup = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["A100 Speedup"].iloc[0]
gh200_stream_copy_spdup = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["GH200 Speedup"].iloc[0]
mi250x_stream_copy_spdup = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["MI250X Speedup"].iloc[0]
mi300a_stream_copy_spdup = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["MI300A Speedup"].iloc[0]

# Set a color palette
sns.set_theme(palette="tab10", font_scale=0.8)
sns.set_style("ticks")
triple_plot_df["color"] = 1
triple_plot_df["Cluster ID"] = relevant_metrics_df["Cluster ID"]

pg_triple = sns.PairGrid(
    data=triple_plot_df,
    y_vars=["Kernel"],
    x_vars=["DDR Memory Bound", "V100 Speedup", "A100 Speedup", "GH200 Speedup", "MI250X Speedup", "MI300A Speedup"],
    despine=True,
    height=10.5,
    aspect=0.25,
    hue="Cluster ID",
    palette=['purple', 'green', 'red', 'darkorange']
)
pg_triple.tight_layout()
pg_triple.map(sns.barplot)
pg_triple.figure.axes[0].set_xlim(0, 1)
pg_triple.figure.axes[0].set_xlabel("SPR-DDR\nMemory Bound")
pg_triple.figure.axes[0].get_yaxis().get_label().set_visible(False)
pg_triple.figure.axes[0].axvline(ddr_stream_copy_mem_bound, color="gold")
pg_triple.figure.axes[1].axvline(1.0, color="lightblue")
pg_triple.figure.axes[1].axvline(v100_stream_copy_spdup, color="gold")
pg_triple.figure.axes[1].set_xlim(0, 10)
pg_triple.figure.axes[1].set_xlabel("P9-V100\nSpeedup")
pg_triple.figure.axes[2].axvline(1.0, color="lightblue")
pg_triple.figure.axes[2].axvline(a100_stream_copy_spdup, color="gold")
pg_triple.figure.axes[2].set_xlim(0, 10)
pg_triple.figure.axes[2].set_xlabel("CAS-A100\nSpeedup")
pg_triple.figure.axes[3].axvline(1.0, color="lightblue")
pg_triple.figure.axes[3].axvline(gh200_stream_copy_spdup, color="gold")
pg_triple.figure.axes[3].set_xlim(0, 10)
pg_triple.figure.axes[3].set_xlabel("GR-GH200\nSpeedup")
pg_triple.figure.axes[4].axvline(1.0, color="lightblue")
pg_triple.figure.axes[4].axvline(mi250x_stream_copy_spdup, color="gold")
pg_triple.figure.axes[4].set_xlim(0, 30)
pg_triple.figure.axes[4].set_xlabel("EPYC-MI250X\nSpeedup")
pg_triple.figure.axes[5].axvline(1.0, color="lightblue")
pg_triple.figure.axes[5].axvline(mi300a_stream_copy_spdup, color="gold")
pg_triple.figure.axes[5].set_xlim(0, 30)
pg_triple.figure.axes[5].set_xlabel("EPYC-MI300A\nSpeedup")

gpu_stream_copy_spdups = {
    "V100": v100_stream_copy_spdup,
    "A100": a100_stream_copy_spdup,
    "GH200": gh200_stream_copy_spdup,
    "MI250X": mi250x_stream_copy_spdup,
    "MI300A": mi300a_stream_copy_spdup,
}

starting_subplot_idx = 1

for i, gpu_type in enumerate(["V100", "A100", "GH200", "MI250X", "MI300A"]):
    for label in pg_triple.figure.axes[0].get_yticklabels():
        _, label_y = label.get_position()
        min_x, max_x = pg_triple.figure.axes[starting_subplot_idx].get_xlim()
        curr_kernel_speedup = triple_plot_df[triple_plot_df["Kernel"] == label.get_text()]["{} Speedup".format(gpu_type)].iloc[0]
        if curr_kernel_speedup > max_x:
            pg_triple.figure.axes[starting_subplot_idx].annotate(
                "{:.2f}".format(curr_kernel_speedup),
                xy=(max_x, label_y),
                xytext=(2, 0),
                textcoords="offset points",
                ha="left",
                va="center",
            )
    starting_subplot_idx+=1

pg_triple_y_ax = pg_triple.figure.axes[0].get_yaxis()
pg_triple_y_ax.set_tick_params(pad=152)
for tick in pg_triple_y_ax.get_majorticklabels():
    tick.set_horizontalalignment("left")
    
pg_triple.savefig("mi300a_diffX.png") 

In [ ]:
# 0 purple, 1 green, 2 red, 3 orange
triple_plot_df = th.statsframe.dataframe[["name", ddr_mem_bound_mean, ddr_to_hbm_speedup, ddr_to_v100_speedup, ddr_to_a100_speedup, ddr_to_gh200_speedup, ddr_to_mi250x_speedup, ddr_to_mi300a_speedup]].reset_index(drop=True)
triple_plot_df.reset_index(inplace=True)

triple_plot_df.columns = triple_plot_df.columns.to_flat_index()
triple_plot_df.rename(
    columns={
        "name": "Kernel",
        ddr_mem_bound_mean: "DDR Memory Bound",
        ddr_to_hbm_speedup: "SPR-HBM Speedup",
        ddr_to_v100_speedup: "V100 Speedup",
        ddr_to_a100_speedup: "A100 Speedup",
        ddr_to_gh200_speedup: "GH200 Speedup",
        ddr_to_mi250x_speedup: "MI250X Speedup",
        ddr_to_mi300a_speedup: "MI300A Speedup",
    },
    inplace=True
)

ddr_stream_copy_mem_bound = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["DDR Memory Bound"].iloc[0]
v100_stream_copy_spdup = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["V100 Speedup"].iloc[0]
a100_stream_copy_spdup = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["A100 Speedup"].iloc[0]
gh200_stream_copy_spdup = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["GH200 Speedup"].iloc[0]
mi250x_stream_copy_spdup = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["MI250X Speedup"].iloc[0]
mi300a_stream_copy_spdup = triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["MI300A Speedup"].iloc[0]

# Set a color palette
sns.set_theme(palette="tab10", font_scale=0.8)
sns.set_style("ticks")
triple_plot_df["color"] = 1
triple_plot_df["Cluster ID"] = relevant_metrics_df["Cluster ID"]

pg_triple = sns.PairGrid(
    data=triple_plot_df,
    y_vars=["Kernel"],
    x_vars=["DDR Memory Bound", "V100 Speedup", "A100 Speedup", "GH200 Speedup", "MI250X Speedup", "MI300A Speedup"],
    despine=True,
    height=10.5,
    aspect=0.25,
    hue="Cluster ID",
    palette=['purple', 'green', 'red', 'darkorange']
)
pg_triple.tight_layout()
pg_triple.map(sns.barplot)
pg_triple.figure.axes[0].set_xlim(0, 1)
pg_triple.figure.axes[0].set_xlabel("SPR-DDR\nMemory Bound")
pg_triple.figure.axes[0].get_yaxis().get_label().set_visible(False)
pg_triple.figure.axes[0].axvline(ddr_stream_copy_mem_bound, color="gold")
pg_triple.figure.axes[1].axvline(1.0, color="lightblue")
pg_triple.figure.axes[1].axvline(v100_stream_copy_spdup, color="gold")
pg_triple.figure.axes[1].set_xlim(0, 50)
pg_triple.figure.axes[1].set_xlabel("P9-V100\nSpeedup")
pg_triple.figure.axes[2].axvline(a100_stream_copy_spdup, color="gold")
pg_triple.figure.axes[2].set_xlim(0, 50)
pg_triple.figure.axes[2].set_xlabel("CAS-A100\nSpeedup")
pg_triple.figure.axes[3].axvline(1.0, color="lightblue")
pg_triple.figure.axes[3].axvline(gh200_stream_copy_spdup, color="gold")
pg_triple.figure.axes[3].set_xlim(0, 50)
pg_triple.figure.axes[3].set_xlabel("GR-GH200\nSpeedup")
pg_triple.figure.axes[4].axvline(1.0, color="lightblue")
pg_triple.figure.axes[4].axvline(mi250x_stream_copy_spdup, color="gold")
pg_triple.figure.axes[4].set_xlim(0, 50)
pg_triple.figure.axes[4].set_xlabel("EPYC-MI250X\nSpeedup")
pg_triple.figure.axes[5].axvline(1.0, color="lightblue")
pg_triple.figure.axes[5].axvline(mi300a_stream_copy_spdup, color="gold")
pg_triple.figure.axes[5].set_xlim(0, 50)
pg_triple.figure.axes[5].set_xlabel("EPYC-MI300A\nSpeedup")

gpu_stream_copy_spdups = {
    "V100": v100_stream_copy_spdup,
    "A100": a100_stream_copy_spdup,
    "GH200": gh200_stream_copy_spdup,
    "MI250X": mi250x_stream_copy_spdup,
    "MI300A": mi300a_stream_copy_spdup,
}

starting_subplot_idx = 1

for i, gpu_type in enumerate(["V100", "A100", "GH200", "MI250X"]):
    for label in pg_triple.figure.axes[0].get_yticklabels():
        _, label_y = label.get_position()
        curr_kernel_speedup = triple_plot_df[triple_plot_df["Kernel"] == label.get_text()]["{} Speedup".format(gpu_type)].iloc[0]
        if curr_kernel_speedup > 1:
            pg_triple.figure.axes[starting_subplot_idx].annotate(
                "{:.2f}".format(curr_kernel_speedup),
                xy=(2, label_y),
                xytext=(15, 0),
                textcoords="offset points",
                ha="left",
                va="center",
            )
    starting_subplot_idx+=1

# starting_subplot_idx = 3
for label in pg_triple.figure.axes[0].get_yticklabels():
    _, label_y = label.get_position()
    min_x, max_x = pg_triple.figure.axes[starting_subplot_idx].get_xlim()
    curr_kernel_speedup = triple_plot_df[triple_plot_df["Kernel"] == label.get_text()]["MI300A Speedup"].iloc[0]
    if curr_kernel_speedup > max_x:
        pg_triple.figure.axes[starting_subplot_idx].annotate(
            "{:.2f}".format(curr_kernel_speedup),
            xy=(max_x, label_y),
            xytext=(2, 0),
            textcoords="offset points",
            ha="left",
            va="center",
        )

pg_triple_y_ax = pg_triple.figure.axes[0].get_yaxis()
pg_triple_y_ax.set_tick_params(pad=152)
for tick in pg_triple_y_ax.get_majorticklabels():
    tick.set_horizontalalignment("left")
    
pg_triple.savefig("mi300a_sameX.png") 

# Boxplots of heatmap summary for FLOPS and Bandwidth

In [ ]:
baseline_bw_df = mem_v_flop_df.reset_index()[mem_v_flop_df["name"].isin(["Basic_MAT_MAT_SHARED", "Stream_TRIAD"])][[
    "name",
    ("ddr", "Memory Bandwidth (GB/sec)_mean"), 
    ("hbm", "Memory Bandwidth (GB/sec)_mean"),
    ("v100", "Memory Bandwidth (GB/sec)_mean"),
    ("a100", "Memory Bandwidth (GB/sec)_mean"), 
    ("gh200", "Memory Bandwidth (GB/sec)_mean"), 
    ("mi250x", "Memory Bandwidth (GB/sec)_mean"),
    ("mi300a", "Memory Bandwidth (GB/sec)_mean"), 
    ]]

baseline_bw_df.columns = [' '.join(col).strip() for col in baseline_bw_df.columns.values]
baseline_bw_df.rename(columns={"n a m e": "name", "ddr Memory Bandwidth (GB/sec)_mean": "SPR-DDR", "hbm Memory Bandwidth (GB/sec)_mean": "SPR-HBM",
                              "v100 Memory Bandwidth (GB/sec)_mean": "4xV100", "a100 Memory Bandwidth (GB/sec)_mean": "2xA100",
                             "gh200 Memory Bandwidth (GB/sec)_mean": "1xGH200", "mi250x Memory Bandwidth (GB/sec)_mean": "8xMI250X",
                             "mi300a Memory Bandwidth (GB/sec)_mean": "4xMI300A"}, inplace=True)
baseline_bw_df = baseline_bw_df.melt(id_vars=["name"], var_name="Architecture", value_name="Achieved Memory Bandwidth (GB/s)")
baseline_bw_df

In [50]:
#sns.set_theme(style='white')
sns.set(font_scale=2, style='white')

In [ ]:
temp_df = mem_v_flop_df[[
    "name",
    ("ddr", "Memory Bandwidth (GB/sec)_mean"),
    ("hbm", "Memory Bandwidth (GB/sec)_mean"),
    ("v100", "Memory Bandwidth (GB/sec)_mean"),
    ("a100", "Memory Bandwidth (GB/sec)_mean"), 
    ("gh200", "Memory Bandwidth (GB/sec)_mean"),
    ("mi250x", "Memory Bandwidth (GB/sec)_mean"),
    ("mi300a", "Memory Bandwidth (GB/sec)_mean"),
]]
temp_df.set_index("name", inplace=True)
bw_boxplot_df = relevant_metrics_df[["Kernel", "Cluster ID"]].join(temp_df, on="Kernel").drop(columns=["Kernel"])
bw_boxplot_df.columns = [' '.join(col).strip() for col in bw_boxplot_df.columns.values]
bw_boxplot_df.rename(columns={"C l u s t e r   I D": "Cluster ID", "ddr Memory Bandwidth (GB/sec)_mean": "SPR-DDR", "hbm Memory Bandwidth (GB/sec)_mean": "SPR-HBM",
                              "v100 Memory Bandwidth (GB/sec)_mean": "4xV100", "a100 Memory Bandwidth (GB/sec)_mean": "2xA100",
                             "gh200 Memory Bandwidth (GB/sec)_mean": "1xGH200", "mi250x Memory Bandwidth (GB/sec)_mean": "8xMI250X",
                             "mi300a Memory Bandwidth (GB/sec)_mean": "4xMI300A"}, inplace=True)
bw_boxplot_df = bw_boxplot_df.melt(id_vars=["Cluster ID"], var_name="Architecture", value_name="Achieved Memory Bandwidth (GB/s)")
bw_boxplot_df

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

sns.boxplot(
    data=bw_boxplot_df,
    x="Architecture",
    y="Achieved Memory Bandwidth (GB/s)",
    hue="Cluster ID",
    ax=ax,
    palette=["tab:blue", "tab:pink"]
)
sns.pointplot(
    data=baseline_bw_df, 
    x="Architecture", 
    y="Achieved Memory Bandwidth (GB/s)",
    hue="name",
    dodge=.4, linestyle="none", errorbar=None,
    marker="_", markersize=70, markeredgewidth=5,
    palette=["tab:orange", "tab:green"],
    legend=True
)
#plt.axhline(1.0, linestyle="--", color="red", label="y = 1")
handles, previous_labels = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=["Compute Bound Cluster", "Memory Bound Cluster", "Basic_MAT_MAT_SHARED", "Stream_TRIAD"], title="", loc="upper left", frameon=False)
plt.ylim(0,16000)
plt.tight_layout()
plt.savefig("bw_clusters.png")
plt.show()

In [ ]:
baseline_flops_df = mem_v_flop_df.reset_index()[mem_v_flop_df["name"].isin(["Basic_MAT_MAT_SHARED", "Stream_TRIAD"])][[
    "name",
    ("ddr", "FLOP rate (GFLOPS)_mean"),
    ("hbm", "FLOP rate (GFLOPS)_mean"),
    ("v100", "FLOP rate (GFLOPS)_mean"),
    ("a100", "FLOP rate (GFLOPS)_mean"),
    ("gh200", "FLOP rate (GFLOPS)_mean"),
    ("mi250x", "FLOP rate (GFLOPS)_mean"), 
    ("mi300a", "FLOP rate (GFLOPS)_mean"), 
    ]]
baseline_flops_df.columns = [' '.join(col).strip() for col in baseline_flops_df.columns.values]
baseline_flops_df.rename(columns={"n a m e": "name", "ddr FLOP rate (GFLOPS)_mean": "SPR-DDR", "hbm FLOP rate (GFLOPS)_mean": "HBM",
                              "v100 FLOP rate (GFLOPS)_mean": "4xV100", "a100 FLOP rate (GFLOPS)_mean": "2xA100",
                             "gh200 FLOP rate (GFLOPS)_mean": "1xGH200", "mi250x FLOP rate (GFLOPS)_mean": "8xMI250X",
                             "mi300a FLOP rate (GFLOPS)_mean": "4xMI300A"}, inplace=True)
baseline_flops_df = baseline_flops_df.melt(id_vars=["name"], var_name="Architecture", value_name="Achieved FLOP Rate (GFLOPS)")
baseline_flops_df

In [ ]:
temp_df = mem_v_flop_df[[
    "name",
    ("ddr", "FLOP rate (GFLOPS)_mean"),
    ("hbm", "FLOP rate (GFLOPS)_mean"),
    ("v100", "FLOP rate (GFLOPS)_mean"),
    ("a100", "FLOP rate (GFLOPS)_mean"),
    ("gh200", "FLOP rate (GFLOPS)_mean"),
    ("mi250x", "FLOP rate (GFLOPS)_mean"),
    ("mi300a", "FLOP rate (GFLOPS)_mean"),
    ]]
temp_df.set_index("name", inplace=True)
flops_boxplot_df = relevant_metrics_df[["Kernel", "Cluster ID"]].join(temp_df, on="Kernel").drop(columns=["Kernel"])
flops_boxplot_df.columns = [' '.join(col).strip() for col in flops_boxplot_df.columns.values]
flops_boxplot_df.rename(columns={"C l u s t e r   I D": "Cluster ID", "ddr FLOP rate (GFLOPS)_mean": "SPR-DDR", "hbm FLOP rate (GFLOPS)_mean": "HBM",
                              "v100 FLOP rate (GFLOPS)_mean": "4xV100", "a100 FLOP rate (GFLOPS)_mean": "2xA100",
                             "gh200 FLOP rate (GFLOPS)_mean": "1xGH200", "mi250x FLOP rate (GFLOPS)_mean": "8xMI250X",
                             "mi300a FLOP rate (GFLOPS)_mean": "4xMI300A"}, inplace=True)
flops_boxplot_df = flops_boxplot_df.melt(id_vars=["Cluster ID"], var_name="Architecture", value_name="Achieved FLOP Rate (GFLOPS)")
flops_boxplot_df

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 10))
sns.boxplot(
    data=flops_boxplot_df,
    x="Architecture",
    y="Achieved FLOP Rate (GFLOPS)",
    hue="Cluster ID",
    ax=ax,
    palette=["tab:blue", "tab:pink"]
)
sns.pointplot(
    data=baseline_flops_df, 
    x="Architecture", 
    y="Achieved FLOP Rate (GFLOPS)",
    hue="name",
    dodge=.4, linestyle="none", errorbar=None,
    marker="_", markersize=70, markeredgewidth=5,
    palette=["tab:orange", "tab:green"],
    legend=True
)
#plt.axhline(1.0, linestyle="--", color="red", label="y = 1")
handles, previous_labels = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=["Compute Bound Cluster", "Memory Bound Cluster", "Basic_MAT_MAT_SHARED", "Stream_TRIAD"], title="", loc="upper left", frameon=False)
plt.ylim(0,18000)
plt.tight_layout()
plt.savefig("flops_clusters.png")
plt.show()

In [ ]:
arch_eff_boxplot_df = relevant_metrics_df[["Kernel", "Cluster ID"]].join(arch_eff_df, on="Kernel").drop(columns=["Kernel"])
arch_eff_boxplot_df.columns = [' '.join(col).strip() for col in arch_eff_boxplot_df.columns.values]
arch_eff_boxplot_df.rename(columns={"C l u s t e r   I D": "Cluster ID"}, inplace=True)
arch_eff_boxplot_df = arch_eff_boxplot_df.melt(id_vars=["Cluster ID"], var_name="Architecture", value_name="Architectural Efficiency Score")
arch_eff_boxplot_df

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 10))
sns.boxplot(
    data=arch_eff_boxplot_df,
    x="Architecture",
    y="Architectural Efficiency Score",
    hue="Cluster ID",
    ax=ax,
    palette=["tab:blue", "tab:pink"]
)
#plt.axhline(1.0, linestyle="--", color="red", label="y = 1")
handles, previous_labels = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=["Compute Bound", "Memory Bound"], title="")
plt.ylim(0,100)
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("arch_eff_clusters.png")
plt.show()

In [ ]:
heatmap_boxplot_df = relevant_metrics_df[["Kernel", "Cluster ID"]].join(perf_port_df, on="Kernel").drop(columns=["Kernel"])
heatmap_boxplot_df = heatmap_boxplot_df.melt(id_vars=["Cluster ID"], var_name="System Set", value_name="Performance Portability Score")
heatmap_boxplot_df

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 10))
sns.boxplot(
    data=heatmap_boxplot_df,
    x="System Set",
    y="Performance Portability Score",
    hue="Cluster ID",
    ax=ax,
    palette=["tab:blue", "tab:pink"]
)
#plt.axhline(1.0, linestyle="--", color="red", label="y = 1")
handles, previous_labels = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=["Compute Bound", "Memory Bound"], title="")
plt.ylim(0,100)
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("perf_port_clusters.png")
plt.show()

# Get Percentage of Stream_TRIAD Performance

In [ ]:
flop_df = th.statsframe.dataframe.reset_index()[["name", ddr_flop_rate_mean, hbm_flop_rate_mean, v100_flop_rate_mean, a100_flop_rate_mean, gh200_flop_rate_mean, mi250x_flop_rate_mean, mi300a_flop_rate_mean, ("label", "")]]
flop_df = flop_df.rename({
    "name": "Kernel",
    ddr_flop_rate_mean: "SPR-DDR FLOP Rate",
    hbm_flop_rate_mean: "SPR-HBM FLOP Rate",
    v100_flop_rate_mean: "V100 FLOP Rate",
    a100_flop_rate_mean: "A100 FLOP Rate",
    gh200_flop_rate_mean: "GH200 FLOP Rate",
    mi250x_flop_rate_mean: "MI250X FLOP Rate",
    mi300a_flop_rate_mean: "MI300A FLOP Rate",
    ("label", ""): "Cluster ID"
}, axis=1)
for i, gpu_type in enumerate(["SPR-DDR", "SPR-HBM", "V100", "A100", "GH200", "MI250X", "MI300A"]):
    flop_df["{} Percent of Basic_MAT_MAT_SHARED".format(gpu_type)] = flop_df["{} FLOP Rate".format(gpu_type)] / flop_df[flop_df["Kernel"] == "Basic_MAT_MAT_SHARED"]["{} FLOP Rate".format(gpu_type)].iloc[0]
flop_df

In [ ]:
bandwidth_df = th.statsframe.dataframe.reset_index()[["name", ddr_mem_bw_mean, hbm_mem_bw_mean, v100_mem_bw_mean, a100_mem_bw_mean, gh200_mem_bw_mean, mi250x_mem_bw_mean, mi300a_mem_bw_mean, ("label", "")]]
bandwidth_df = bandwidth_df.rename({
    "name": "Kernel",
    ddr_mem_bw_mean: "SPR-DDR Memory Bandwidth",
    hbm_mem_bw_mean: "SPR-HBM Memory Bandwidth",
    v100_mem_bw_mean: "V100 Memory Bandwidth",
    a100_mem_bw_mean: "A100 Memory Bandwidth",
    gh200_mem_bw_mean: "GH200 Memory Bandwidth",
    mi250x_mem_bw_mean: "MI250X Memory Bandwidth",
    mi300a_mem_bw_mean: "MI300A Memory Bandwidth",
    ("label", ""): "Cluster ID"
}, axis=1)
for i, gpu_type in enumerate(["SPR-DDR", "SPR-HBM", "V100", "A100", "GH200", "MI250X", "MI300A"]):
    bandwidth_df["{} Percent of Stream_TRIAD".format(gpu_type)] = bandwidth_df["{} Memory Bandwidth".format(gpu_type)] / bandwidth_df[bandwidth_df["Kernel"] == "Stream_TRIAD"]["{} Memory Bandwidth".format(gpu_type)].iloc[0]
bandwidth_df

In [ ]:
for i, gpu_type in enumerate(["SPR-HBM", "V100", "A100", "GH200", "MI250X", "MI300A"]):
    triple_plot_df["{} Percent of Stream_TRIAD".format(gpu_type)] = triple_plot_df["{} Speedup".format(gpu_type)] / triple_plot_df[triple_plot_df["Kernel"] == "Stream_TRIAD"]["{} Speedup".format(gpu_type)].iloc[0]

triple_plot_df

In [63]:
pd.set_option("display.max_columns", None)

In [ ]:
speedup_df = triple_plot_df.drop(["Kernel", "index", "color"], axis="columns").groupby("Cluster ID").agg([np.min, np.mean, np.max])
speedup_df

In [65]:
size=20
plt.rcParams.update({
    'font.size': size,         # General font size
    'axes.titlesize': size,    # Title size of axes
    'axes.labelsize': size,    # Size of axis labels
    'xtick.labelsize': size,   # X-axis tick labels size
    'ytick.labelsize': size,   # Y-axis tick labels size
    'legend.fontsize': size,   # Legend font size
    'legend.title_fontsize': size,
    'figure.titlesize': size   # Figure title size
})

# min/max/ave speedup for each cluster, on each platform

In [ ]:
df = speedup_df.loc[:, ["SPR-HBM Speedup", "V100 Speedup", "A100 Speedup", "GH200 Speedup", "MI250X Speedup", "MI300A Speedup"]].T

# Reshape the dataframe for visualization
df = df.stack().reset_index()
df.columns = ['GPU', 'Metric', 'Cluster', 'Speedup']

df = df.replace("Speedup", "", regex=True)

# Create separate plots for each cluster
clusters = df['Cluster'].unique()
fig, axes = plt.subplots(1, len(clusters), figsize=(15, 6), sharey=True)

plt.ylim(bottom=-1, top=47)

for i, cluster in enumerate(clusters):
    cluster_data = df[df['Cluster'] == cluster]
    sns.barplot(x="GPU", y="Speedup", hue="Metric", data=cluster_data, ax=axes[i])
    if i == 0:
        axes[i].set_title(f"Compute Bound Cluster")
    elif i == 1:
        axes[i].set_title(f"Memory Bound Cluster")
    axes[i].set_ylabel("Speedup")
    axes[i].set_xlabel("Architecture")
    #axes[i].tick_params(axis='x', rotation=45)
    #axes[i].grid(axis="y", linestyle="--", alpha=0.7)
    axes[i].legend(title="Metric")

    ax = axes[i]
    for container in ax.containers:
        for bar in container:
            height = bar.get_height()
            
            # If the bar height is less than 1, add the text above the bar
            #if height < 1:
            ax.text(bar.get_x() + bar.get_width() / 2, height + 0.05,  # Adjust text position
                    f'{height:.2f}',  # Show the speedup value (you can format as needed)
                    ha='center', va='bottom', fontsize=10, color='black')

plt.suptitle("DDR Speedup (Runtime) Comparison Across Clusters")
plt.tight_layout()
plt.show()

In [ ]:
box_data = triple_plot_df.drop(["Kernel", "index", "color"], axis="columns")
box_data = box_data.loc[:, ["SPR-HBM Speedup", "V100 Speedup", "A100 Speedup", "GH200 Speedup", "MI250X Speedup", "MI300A Speedup", "Cluster ID"]]
box_data = box_data.rename(columns={
    "SPR-HBM Speedup": "SPR-HBM",
    "V100 Speedup": "4xV100",
    "A100 Speedup": "2xA100",
    "GH200 Speedup": "1xGH200",
    "MI250X Speedup": "8xMI250X",
    "MI300A Speedup": "4xMI300A"
})
box_data = box_data.melt(id_vars=["Cluster ID"], var_name="Architecture", value_name="Speedup")
box_data

In [ ]:
baseline_speedup_df = triple_plot_df.drop(["index", "color"], axis="columns")[triple_plot_df["Kernel"].isin(["Basic_MAT_MAT_SHARED", "Stream_TRIAD"])]
baseline_speedup_df = baseline_speedup_df.loc[:, ["Kernel", "SPR-HBM Speedup", "V100 Speedup", "A100 Speedup", "GH200 Speedup", "MI250X Speedup", "MI300A Speedup", "Cluster ID"]]
baseline_speedup_df = baseline_speedup_df.rename(columns={
    "SPR-HBM Speedup": "SPR-HBM",
    "V100 Speedup": "4xV100",
    "A100 Speedup": "2xA100",
    "GH200 Speedup": "1xGH200",
    "MI250X Speedup": "8xMI250X",
    "MI300A Speedup": "4xMI300A"
})
baseline_speedup_df = baseline_speedup_df.melt(id_vars=["Kernel", "Cluster ID"], var_name="Architecture", value_name="Speedup")
baseline_speedup_df

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 10))
sns.boxplot(
    data=box_data,
    x="Architecture",
    y="Speedup",
    hue="Cluster ID",
    ax=ax,
    palette=["tab:blue", "tab:pink"]
)
sns.pointplot(
    data=baseline_speedup_df, 
    x="Architecture", 
    y="Speedup",
    hue="Kernel",
    dodge=.4, linestyle="none", errorbar=None,
    marker="_", markersize=70, markeredgewidth=5,
    palette=["tab:orange", "tab:green"],
    legend=True
)
plt.axhline(1.0, linestyle="--", color="red", label="y = 1")
handles, previous_labels = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=["Compute Bound Cluster", "Memory Bound Cluster", "Basic_MAT_MAT_SHARED", "Stream_TRIAD", "y = 1"],frameon=False)
plt.tight_layout()
plt.savefig("speedup_stats.png")
plt.show()

# for the memory bound ones, what % of stream speedup does each one get on each platform?

In [ ]:
raw_bw_df = bandwidth_df[bandwidth_df["Cluster ID"] == 1].loc[:, [
    "Kernel",
    "SPR-DDR Memory Bandwidth",
    "SPR-HBM Memory Bandwidth",
    "V100 Memory Bandwidth", 
    "A100 Memory Bandwidth", 
    "GH200 Memory Bandwidth", 
    "MI250X Memory Bandwidth", 
    "MI300A Memory Bandwidth"
    ]]
raw_bw_df

In [ ]:
raw_bw_df_compbound = bandwidth_df[bandwidth_df["Cluster ID"] == 0].loc[:, [
    "Kernel",
    "SPR-DDR Memory Bandwidth",
    "SPR-HBM Memory Bandwidth",
    "V100 Memory Bandwidth", 
    "A100 Memory Bandwidth", 
    "GH200 Memory Bandwidth", 
    "MI250X Memory Bandwidth", 
    "MI300A Memory Bandwidth"
    ]]
raw_bw_df_compbound

In [ ]:
perc_triad_df = bandwidth_df[bandwidth_df["Cluster ID"] == 1].loc[:, [
    "Kernel",
    "SPR-DDR Percent of Stream_TRIAD",
    "SPR-HBM Percent of Stream_TRIAD",
    "V100 Percent of Stream_TRIAD", 
    "A100 Percent of Stream_TRIAD", 
    "GH200 Percent of Stream_TRIAD", 
    "MI250X Percent of Stream_TRIAD", 
    "MI300A Percent of Stream_TRIAD"
    ]]
perc_triad_df

In [73]:
size=20
plt.rcParams.update({
    'font.size': size,         # General font size
    'axes.titlesize': size,    # Title size of axes
    'axes.labelsize': 19,    # Size of axis labels
    'xtick.labelsize': size,   # X-axis tick labels size
    'ytick.labelsize': 15,   # Y-axis tick labels size
    'legend.fontsize': size,   # Legend font size
    'legend.title_fontsize': size,
    'figure.titlesize': size   # Figure title size
})

In [74]:
num_map = {
    "V100": "4xV100",
    "A100": "2xA100",
    "GH200": "1xGH200",
    "MI250X": "8xMI250X",
    "MI300A": "4xMI300A"
}

In [ ]:
# Reshape the data to long format
df_long = pd.melt(raw_bw_df, id_vars=["Kernel"], var_name="Architecture", value_name="Memory Bandwidth")

# Extract architecture from column name
df_long['Architecture'] = df_long['Architecture'].str.replace(" Memory Bandwidth", "")

df_long = df_long.replace(num_map)

# Plot
plt.figure(figsize=(10, 15))
ax = sns.barplot(
    x="Memory Bandwidth", 
    y="Kernel", 
    hue="Architecture", 
    data=df_long, 
    orient="h"
)
plt.xlabel("Memory Bandwidth (GB/s)")
plt.ylabel("")

lim=10000
# Cap the x-axis at 200
plt.xlim(0, lim)

# Annotate values greater than lim
for i, row in df_long.iterrows():
    if row["Memory Bandwidth"] > lim:
        x_val = min(row["Memory Bandwidth"], lim)
        mod = len(set(df_long["Kernel"]))
        ax.text(
            x_val, i%mod+(i//mod*0.3)-1.25, 
            f"{row['Memory Bandwidth']:.1f}", 
            #color='red', 
            #va='bottom', 
            fontsize=10
        )
    
plt.legend(bbox_to_anchor=(.7, 0.5))
#plt.axvline(x=100, color='black', linestyle='--', label="x = 100")
sns.despine()
plt.tight_layout()
plt.show()
plt.clf()


In [ ]:
# Reshape the data to long format
df_long = pd.melt(raw_bw_df_compbound, id_vars=["Kernel"], var_name="Architecture", value_name="Memory Bandwidth")

# Extract architecture from column name
df_long['Architecture'] = df_long['Architecture'].str.replace(" Memory Bandwidth", "")

df_long = df_long.replace(num_map)

# Plot
plt.figure(figsize=(10, 15))
ax = sns.barplot(
    x="Memory Bandwidth", 
    y="Kernel", 
    hue="Architecture", 
    data=df_long, 
    orient="h"
)
plt.xlabel("Memory Bandwidth (GB/s)")
plt.ylabel("")

lim=10000
# Cap the x-axis at 200
plt.xlim(0, lim)

# Annotate values greater than lim
for i, row in df_long.iterrows():
    if row["Memory Bandwidth"] > lim:
        x_val = min(row["Memory Bandwidth"], lim)
        mod = len(set(df_long["Kernel"]))
        ax.text(
            x_val, i%mod+(i//mod*0.3)-1.25, 
            f"{row['Memory Bandwidth']:.1f}", 
            #color='red', 
            #va='bottom', 
            fontsize=10
        )
    
plt.legend(bbox_to_anchor=(.7, 0.5))
#plt.axvline(x=100, color='black', linestyle='--', label="x = 100")
sns.despine()
plt.tight_layout()
plt.show()
plt.clf()


In [ ]:
# Reshape the data to long format
df_long = pd.melt(perc_triad_df, id_vars=["Kernel"], var_name="Architecture", value_name="Percent of Stream_TRIAD")
df_long["Percent of Stream_TRIAD"] = df_long["Percent of Stream_TRIAD"] * 100

# Extract architecture from column name
df_long['Architecture'] = df_long['Architecture'].str.replace(" Percent of Stream_TRIAD", "")

df_long = df_long.replace(num_map)

# Plot
plt.figure(figsize=(10, 20))
ax = sns.barplot(
    x="Percent of Stream_TRIAD", 
    y="Kernel", 
    hue="Architecture", 
    data=df_long, 
    orient="h"
)
plt.xlabel("Percent Mem BW / Stream_TRIAD Mem BW")
           #Percent Speedup (Mem BW) over Stream_TRIAD")
plt.ylabel("")

# Cap the x-axis at 200
plt.xlim(0, 200)

# Annotate values greater than 200

for i, row in df_long.iterrows():
    if row["Percent of Stream_TRIAD"] > 175:
        x_val = min(row["Percent of Stream_TRIAD"], 205)
        mod = len(set(df_long["Kernel"]))
        ax.text(
            x_val, i%mod+(i//mod*0.2)-0.3, 
            f"{row['Percent of Stream_TRIAD']:.1f}%", 
            #color='red', 
            #va='bottom', 
            fontsize=10
        )
    
plt.legend(bbox_to_anchor=(.7, 0.5))
plt.axvline(x=100, color='black', linestyle='--', label="x = 100")
sns.despine()
plt.tight_layout()
plt.show()
plt.clf()


# for the memory bound ones, which set is roughly the same speedup as stream triad, which set is significantly lower, and which set is higher - per platform

In [ ]:
low_list = []
middle_list = []
high_list = []
lowbound = 0.75
highbound = 1.25
for arch in set(df_long["Architecture"].values):
    print(arch)
    tdf = df_long[df_long["Architecture"] == arch]

    lowset = set(tdf[tdf["Percent of Stream_TRIAD"] < lowbound]["Kernel"])
    middleset = set(tdf[(tdf["Percent of Stream_TRIAD"] > lowbound) & (tdf["Percent of Stream_TRIAD"] < highbound)]["Kernel"])
    highset = set(tdf[tdf["Percent of Stream_TRIAD"] > highbound]["Kernel"])

    print(f"\t< {lowbound}", lowset)
    print(f"\tmiddle", middleset)
    print(f"\t> {highbound}", highset)

    low_list.append(lowset)
    middle_list.append(middleset)
    high_list.append(highset)

# for the memory bound ones, which set has roughly the same % speedup relative to stream triad on ALL platforms?

In [ ]:
print(set.intersection(*middle_list))

# for the memory bound ones, which set has good speedup on a subset of the platforms - and shows less speedup on others? 

In [80]:
df = df_long
# Classify the Percent of Stream_TRIAD into ranges
bins = [-float('inf'), 75, 125, float('inf')]
labels = ['75%', '75% - 125%', '> 125%']
df['Stream_TRIAD_Range'] = pd.cut(df['Percent of Stream_TRIAD'], bins=bins, labels=labels)

# Group by Kernel and Stream_TRIAD_Range, then count occurrences
result = df.groupby(['Kernel', 'Stream_TRIAD_Range']).size().reset_index(name='Architectures')


In [ ]:
result

In [ ]:
plt.figure(figsize=(5, 15))
sns.barplot(
    data=result,
    x="Architectures",
    y="Kernel",
    hue="Stream_TRIAD_Range",
)
plt.show()

# for the retiring cluster, what % of matmat speedup does each one get?

In [ ]:
for i, gpu_type in enumerate(["V100", "A100", "GH200", "MI250X", "MI300A"]):
    triple_plot_df["{} Percent of Basic_MAT_MAT_SHARED".format(gpu_type)] = triple_plot_df["{} Speedup".format(gpu_type)] / triple_plot_df[triple_plot_df["Kernel"] == "Basic_MAT_MAT_SHARED"]["{} Speedup".format(gpu_type)].iloc[0]

triple_plot_df

In [ ]:
flop_df

In [ ]:
raw_flop_df = flop_df[flop_df["Cluster ID"] == 0].loc[:, [
    "Kernel",
    "SPR-DDR FLOP Rate",
    "SPR-HBM FLOP Rate",
    "V100 FLOP Rate", 
    "A100 FLOP Rate", 
    "GH200 FLOP Rate", 
    "MI250X FLOP Rate", 
    "MI300A FLOP Rate"
    ]]
raw_flop_df

In [ ]:
raw_flop_df_membound = flop_df[flop_df["Cluster ID"] == 1].loc[:, [
    "Kernel",
    "SPR-DDR FLOP Rate",
    "SPR-HBM FLOP Rate",
    "V100 FLOP Rate", 
    "A100 FLOP Rate", 
    "GH200 FLOP Rate", 
    "MI250X FLOP Rate", 
    "MI300A FLOP Rate"
    ]]
raw_flop_df_membound

In [ ]:
perc_mat_mat_df = flop_df[flop_df["Cluster ID"] == 0].loc[:, [
    "Kernel",
    "SPR-DDR Percent of Basic_MAT_MAT_SHARED",
    "SPR-HBM Percent of Basic_MAT_MAT_SHARED",
    "V100 Percent of Basic_MAT_MAT_SHARED", 
    "A100 Percent of Basic_MAT_MAT_SHARED", 
    "GH200 Percent of Basic_MAT_MAT_SHARED", 
    "MI250X Percent of Basic_MAT_MAT_SHARED", 
    "MI300A Percent of Basic_MAT_MAT_SHARED"
    ]]
perc_mat_mat_df

In [88]:
size=20
plt.rcParams.update({
    'font.size': size,         # General font size
    'axes.titlesize': size,    # Title size of axes
    'axes.labelsize': 19,    # Size of axis labels
    'xtick.labelsize': size,   # X-axis tick labels size
    'ytick.labelsize': 15,   # Y-axis tick labels size
    'legend.fontsize': size,   # Legend font size
    'legend.title_fontsize': size,
    'figure.titlesize': size   # Figure title size
})

In [ ]:
# Reshape the data to long format
df_long = pd.melt(raw_flop_df, id_vars=["Kernel"], var_name="Architecture", value_name="FLOP Rate")

# Extract architecture from column name
df_long['Architecture'] = df_long['Architecture'].str.replace(" FLOP Rate", "")

df_long = df_long.replace(num_map)

# Plot
fig, ax = plt.subplots(figsize=(9, 15))
sns.barplot(x="FLOP Rate", y="Kernel", hue="Architecture", data=df_long, orient="h")
plt.xlabel("FLOP Rate (GFLOPS)")
plt.ylabel("")

lim=7500
# Cap the x-axis at 200
plt.xlim(0, lim)

# Annotate values greater than lim
for i, row in df_long.iterrows():
    if row["FLOP Rate"] > lim:
        x_val = min(row["FLOP Rate"], lim)
        mod = len(set(df_long["Kernel"]))
        ax.text(
            x_val, i%mod+(i//mod*0.2)-.7, 
            f"{row['FLOP Rate']:.1f}", 
            #color='red', 
            #va='bottom', 
            fontsize=10
        )
#plt.tight_layout()
sns.despine()
plt.legend(title="")
plt.show()
plt.clf()

In [ ]:
# Reshape the data to long format
df_long = pd.melt(raw_flop_df_membound, id_vars=["Kernel"], var_name="Architecture", value_name="FLOP Rate")

# Extract architecture from column name
df_long['Architecture'] = df_long['Architecture'].str.replace(" FLOP Rate", "")

df_long = df_long.replace(num_map)

# Plot
fig, ax = plt.subplots(figsize=(9, 15))
sns.barplot(x="FLOP Rate", y="Kernel", hue="Architecture", data=df_long, orient="h")
plt.xlabel("FLOP Rate (GFLOPS)")
plt.ylabel("")

lim=7500
# Cap the x-axis at 200
plt.xlim(0, lim)

# Annotate values greater than lim
for i, row in df_long.iterrows():
    if row["FLOP Rate"] > lim:
        x_val = min(row["FLOP Rate"], lim)
        mod = len(set(df_long["Kernel"]))
        ax.text(
            x_val, i%mod+(i//mod*0.2)-.7, 
            f"{row['FLOP Rate']:.1f}", 
            #color='red', 
            #va='bottom', 
            fontsize=10
        )
#plt.tight_layout()
sns.despine()
plt.legend(title="")
plt.show()
plt.clf()

In [ ]:
# Reshape the data to long format
df_long = pd.melt(perc_mat_mat_df, id_vars=["Kernel"], var_name="Architecture", value_name="Percent of Basic_MAT_MAT_SHARED")
df_long["Percent of Basic_MAT_MAT_SHARED"] = df_long["Percent of Basic_MAT_MAT_SHARED"] * 100

# Extract architecture from column name
df_long['Architecture'] = df_long['Architecture'].str.replace(" Percent of Basic_MAT_MAT_SHARED", "")

df_long = df_long.replace(num_map)

# Plot
plt.figure(figsize=(10, 20))
sns.barplot(x="Percent of Basic_MAT_MAT_SHARED", y="Kernel", hue="Architecture", data=df_long, orient="h")
#plt.title("Percent of Basic_MAT_MAT_SHARED for Compute Bound Cluster")
plt.xlabel("Percent FLOPS / Basic_MAT_MAT_SHARED FLOPS")
plt.ylabel("")
plt.axvline(x=100, color='black', linestyle='--', label="x = 100")
plt.xlim(0, 200)
#plt.tight_layout()
plt.legend(title="")
sns.despine()
plt.show()

# for the retiring cluster, which set gets speedup higher than stream - and on which platforms?

In [ ]:
bandwidth_df

In [ ]:
retiring_stream_df = triple_plot_df[(triple_plot_df["Cluster ID"] == 0) | (triple_plot_df["Kernel"] == "Stream_TRIAD")][[
    "Kernel",
    "V100 Speedup",
    "A100 Speedup",
    "GH200 Speedup",
    "MI250X Speedup",
    "MI300A Speedup"]
]
retiring_stream_df

In [ ]:
df = retiring_stream_df

# Find the value of Stream_TRIAD for each column
stream_triad = df[df['Kernel'] == "Stream_TRIAD"].iloc[:, 1:]

# Filter rows with values greater than Stream_TRIAD
filtered_df = df[df.iloc[:, 1:].gt(stream_triad.values).any(axis=1)]

# Display the filtered DataFrame
stream_triad["Kernel"] = "Stream_TRIAD"
tdf = pd.concat([filtered_df,stream_triad])
tdf

In [ ]:
# Reshape the data to long format
df_long = pd.melt(tdf, id_vars=["Kernel"], var_name="Architecture", value_name="Speedup > Stream_TRIAD")

# Extract architecture from column name
df_long['Architecture'] = df_long['Architecture'].str.replace(" Speedup > Stream_TRIAD", "")

# Plot
plt.figure(figsize=(5, 15))
sns.barplot(x="Speedup > Stream_TRIAD", y="Kernel", hue="Architecture", data=df_long, orient="h")
plt.title("Speedup > Stream_TRIAD for Compute Bound Cluster")
plt.xlabel("Speedup > Stream_TRIAD")
plt.ylabel("Kernel")
plt.axvline(x=1, color='black', linestyle='--', label="x = 1")
#plt.tight_layout()
plt.show()

In [ ]:
tdf

In [ ]:
# Reshape the data to long format
df_long = pd.melt(tdf, id_vars=["Kernel"], var_name="Architecture", value_name="Speedup")

# Extract Stream_TRIAD values for each architecture
stream_triads = df_long[df_long['Kernel'] == 'Stream_TRIAD'][['Architecture', 'Speedup']].set_index('Architecture')

# Merge the Stream_TRIAD values with the original dataframe
df_long = df_long.merge(stream_triads, on='Architecture', suffixes=('', '_Stream_TRIAD'))

# Add a 'Color' column based on whether the value is higher than the corresponding Stream_TRIAD value
df_long['Color'] = df_long.apply(lambda row: 'orange' if row['Speedup'] > row['Speedup_Stream_TRIAD'] else 'green', axis=1)

# Plot using FacetGrid
g = sns.FacetGrid(df_long, col="Architecture", col_wrap=5, height=5, sharey=True)
g.map(sns.barplot, "Speedup", "Kernel", data=df_long, hue="Color", orient="h", palette=["tab:blue", "tab:red"])

# Add vertical line at x = 1
for i, ax in enumerate(g.axes.flat):
    ax.axvline(x=tdf[tdf["Kernel"] == "Stream_TRIAD"].T.iloc[i+1].iloc[0], color='black', linestyle='--', label="x = 1")
    ax.set_title(ax.get_title(), fontsize=12)

g.set_axis_labels("Speedup", "Kernel")
g.set_titles(col_template="{col_name}")

plt.subplots_adjust(hspace=0.5)
plt.show()

## Kernels that didn't achieve Stream_TRIAD on MI300A but did on other GPUs

In [ ]:
memory_bound_df = bandwidth_df[(bandwidth_df["Cluster ID"] == 0) | (bandwidth_df["Cluster ID"] == 1)]

slow_kernels = {"V100": [], "MI250X": [], "MI300A": []}
for i, gpu_type in enumerate(["V100", "MI250X", "MI300A"]): 
    for index, _ in memory_bound_df.iterrows():
        if memory_bound_df["{} Percent of Stream_TRIAD".format(gpu_type)].loc[index] < 1:
            kernel = memory_bound_df["Kernel".format(gpu_type)].loc[index]
            slow_kernels[gpu_type].append(kernel)
            
slow_kernels

In [ ]:
v100_mi300a = set(slow_kernels["MI300A"]) - set(slow_kernels["V100"])
mi250x_mi300a =  set(slow_kernels["MI300A"]) - set(slow_kernels["MI250X"])

print(v100_mi300a)
print(mi250x_mi300a)

#### 6.3D Parallel Coordinate Plots

Note: these plots are created with the plotly library which may require updates to your jupyter environment, and they will not render in LLNL's LC jupyter environment

If experiencing rendering issues, try the following command followed by restarting the kernel.
```
pip install --upgrade nbformat
```

In [ ]:
# Get the averages
avg_no_outliers = relevant_metrics_df.set_index("Kernel").groupby("Cluster ID", as_index=False).mean()
avg_no_outliers.head()

In [101]:
# fig = go.Figure(data=
#     # 
#     go.Parcoords(
#         line = dict(color = avg_no_outliers["Cluster ID"],
#                    colorscale = [[0, "purple"], [0.33, "green"], [0.66, "red"], [1, "darkorange"],]),
#         dimensions = list([
#             dict(range = [0,3],
#                  tickvals = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
#                 label = "Bad Speculation", values = avg_no_outliers["Bad Speculation"]),
#             dict(range = [0,3],
#                  tickvals = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
#                 label = "Frontend Bound", values = avg_no_outliers["Frontend Bound"]),
#             dict(range = [0,3],
#                  tickvals = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
#                 label = "Core Bound", values = avg_no_outliers["Core Bound"]),
#             dict(range = [0,3],
#                  tickvals = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
#                 label = "Memory Bound", values = avg_no_outliers["Memory Bound"]),
#             dict(range = [0,3],
#                  tickvals = [0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1],
#                 label = "Retiring", values = avg_no_outliers["Retiring"]),
#             dict(range = [-10,22],
#                  tickvals = [0,2,4,],
#                 label = "SPR-HBM Speedup", values = avg_no_outliers["Speedup on SPR-HBM"]),
#             dict(range = [-10,22],
#                  tickvals = [0,2,4,6,8,10],
#                 label = "P9-V100 Speedup", values = avg_no_outliers["Speedup on P9-V100"]),
#             dict(range = [-10,22],
#                  tickvals = [0,2,4,6,8,10,12,14,16,18,20,22],
#                 label = "EPYC-MI250X Speedup", values = avg_no_outliers["Speedup on EPYC-MI250X"])
#         ])
#     ),
#     #Custom Legend
#     go.Scatter(
#             x=[None],
#             y=[None],
#             mode="lines",
#             name="0",
#             marker=dict(size=7, color="purple", ),
#         ),

#         go.Scatter(
#             x=[None],
#             y=[None],
#             mode="lines",
#             name="1",
#             marker=dict(size=7, color="green", ),
#         ),

#         go.Scatter(
#             x=[None],
#             y=[None],
#             mode="lines",
#             name="2",
#             marker=dict(size=7, color="red",),
#         ),
#         go.Scatter(
#             x=[None],
#             y=[None],
#             mode="lines",
#             name="3",
#             marker=dict(size=7, color="darkorange",),
#         ),               
#     ]   
# )

# fig.update_layout(
#     width=1100, height=750,
#     plot_bgcolor = "white",
#     paper_bgcolor = "white"
# )
# fig.update_xaxes(showticklabels=False)
# fig.update_yaxes(showticklabels=False)
# fig.update_layout(
#     legend=dict(
#         title="Cluster",
#         orientation="h",
#         yanchor="auto",
#         y=0.9,
#         xanchor="right",
#         x=0.3
#     )
# )

# #fig.write_image("images/par_coord_AVG_NO_OUTLIERS.png")

# fig.show()

In [102]:
# # plot all kernels
# fig = go.Figure(data=
#     [
#         go.Parcoords(
#             line = dict(
#                 color = relevant_metrics_df["Cluster ID"], 
#                 colorscale = [[0, "purple"], [0.33, "green"], [0.66, "red"], [1, "darkorange"],]
#             ),
#             dimensions = list([
#                 dict(
#                     range = [0, 5],
#                     tickvals = [0, 0.2, 0.4, 0.6, 0.8, 1],
#                     label = "Bad Speculation",
#                     values = relevant_metrics_df["Bad Speculation"],
#                 ),
#                 dict(
#                     range = [0, 5],
#                     tickvals = [0, 0.2, 0.4, 0.6, 0.8, 1],
#                     label = "Frontend Bound",
#                     values = relevant_metrics_df["Frontend Bound"],
#                 ),
#                 dict(
#                     range = [0, 5],
#                     tickvals = [0, 0.2, 0.4, 0.6, 0.8, 1],
#                     label = "Core Bound",
#                     values = relevant_metrics_df["Core Bound"],
#                 ),
#                 dict(
#                     range = [0, 5],
#                     tickvals = [0, 0.2, 0.4, 0.6, 0.8, 1],
#                     label = "Memory Bound",
#                     values = relevant_metrics_df["Memory Bound"],
#                 ),
#                 dict(
#                     range = [0, 5],
#                     tickvals = [0, 0.2, 0.4, 0.6, 0.8, 1],
#                     label = "Retiring",
#                     values = relevant_metrics_df["Retiring"],
#                 ),
#                 dict(
#                     range = [-10, 40],
#                     tickvals = [0, 5],
#                     label = "SPR-HBM Speedup",
#                     values = relevant_metrics_df["Speedup on SPR-HBM"],
#                 ),
#                 dict(
#                     range = [-10, 40],
#                     tickvals = [0, 5, 10, 15, 20],
#                     label = "P9-V100 Speedup",
#                     values = relevant_metrics_df["Speedup on P9-V100"],
#                 ),
#                 dict(
#                     range = [-10, 40],
#                     tickvals = [0, 10, 20, 30, 40],
#                     label = "EPYC-MI250X Speedup",
#                     values = relevant_metrics_df["Speedup on EPYC-MI250X"],
#                 )
#             ])
#         ),
#         #Custom Legend
#         go.Scatter(
#             x=[None],
#             y=[None],
#             mode="lines",
#             name="0",
#             marker=dict(size=7, color="purple"),
#         ),

#         go.Scatter(
#             x=[None],
#             y=[None],
#             mode="lines",
#             name="1",
#             marker=dict(size=7, color="green"),
#         ),

#         go.Scatter(
#             x=[None],
#             y=[None],
#             mode="lines",
#             name="2",
#             marker=dict(size=7, color="red"),
#         ),
#         go.Scatter(
#             x=[None],
#             y=[None],
#             mode="lines",
#             name="3",
#             marker=dict(size=7, color="darkorange"),
#         ),                    
#     ]   
# )

# fig.update_layout(
#     width=900,
#     height=1000,
#     plot_bgcolor = "white",
#     paper_bgcolor = "white",
# )

# fig.update_xaxes(showticklabels=False)
# fig.update_yaxes(showticklabels=False)
# fig.update_layout(
#     legend=dict(
#         title="Cluster",
#         orientation="h",
#         yanchor="auto",
#         y=0.9,
#         xanchor="right",
#         x=0.4,
#     )
# )

# #fig.write_image("images/par_coord_NO_OUTLIERS.png")

# fig.show()